In [1]:
import heapq
import random
from bisect import bisect_left
from collections import Counter, defaultdict, deque

# 1) Binary Tree DFS

**In plain English.** A tree question is a recursion question. Each node asks its children
a question, combines their answers with its own value, and reports one answer to its
parent.

> **Analogy.** A company org chart. To count everyone, each manager asks each direct
> report "how many people under you?", adds them up, adds one for themself, and reports
> to their own boss. Nobody needs to see the whole chart.

The frame for every question:

> *What does a node need from its children, and what does it return to its parent?*

| Style | When | Shape |
|---|---|---|
| **Bottom-up** (post-order) | answer depends on subtrees: height, diameter, balanced, max path | `left = f(node.left); right = f(node.right); combine` |
| **Top-down** (pre-order) | state flows from the root: path sum, max so far, current number | `f(node, state)` passes updated state down |
| **Global best + local return** | the best answer bends through a node, but only one branch can continue upward | return one branch, record `left + right` in a nonlocal |

The helper cell below builds trees from LeetCode's level-order list (`None` = missing
child) and converts them back, so tests read like the problem statements.

**Recognition signals**

- "depth", "height", "balanced", "diameter", "same / symmetric / subtree"
- "path sum", "root-to-leaf", "any path", "good nodes", "max so far on the path"
- "lowest common ancestor"
- "construct the tree from traversals", "serialize", "flatten"
- adjacency constraints on a tree ("cannot rob parent and child") → tree DP returning a tuple

**Common bugs**

- Treating a missing child as a leaf (Minimum Depth, Path Sum): a leaf has *no* children.
- Returning the global best instead of the one-branch value to the parent (Max Path Sum).
- Mutating a shared `path` list without undoing it (backtrack with `pop()`).
- Recomputing heights at every node → O(n²); return height and status together.
- Deep skewed trees exceed Python's recursion limit (~1000): use an explicit stack.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Tree BFS | processes level by level with a queue; DFS goes deep first via recursion |
| BST algorithms | use the ordering to skip half the tree; plain tree DFS must visit everything |
| Graph DFS | graphs need a `visited` set (cycles); trees never revisit a node |
| Backtracking | explores *choices*; tree DFS explores an existing structure (but path collection uses the same append/pop) |

**Learning path.** Height (the base recursion) → comparisons (Same, Symmetric, Subtree) →
transform (Invert) → global best (Diameter, Balanced, Max Path Sum) → LCA → flow along
edges (Coins) → tree DP with tuples (House Robber III) → greedy states (Cameras). Then
top-down state (Path Sum family, Good Nodes, ZigZag), and finally building and
serialising trees.

In [2]:
class TreeNode:
    """Binary tree node."""

    def __init__(
        self,
        val: int = 0,
        left: "TreeNode | None" = None,
        right: "TreeNode | None" = None,
    ) -> None:
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> TreeNode | None:
    """Tree from LeetCode level-order list with None gaps."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        for side in ("left", "right"):
            if i < len(values) and values[i] is not None:
                child = TreeNode(values[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root


def tree_to_list(root: TreeNode | None) -> list:
    """LeetCode level-order list of a tree (trailing None trimmed)."""
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node:
            out.append(node.val)
            queue.extend([node.left, node.right])
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


def find(root: TreeNode | None, val: int) -> TreeNode | None:
    """First node with the given value (DFS)."""
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)


print(tree_to_list(build_tree([3, 9, 20, None, None, 15, 7])))

[3, 9, 20, None, None, 15, 7]


## 1.1 Bottom-up recursion

Bottom-up: recurse first, decide after. Write the return value in words before coding
("height of this subtree", "best downward chain", "(robbed, skipped)"). If the answer can
pass *through* a node, keep it in a nonlocal `best`; return only what a parent can use.

### 1.1.1 [104. Maximum Depth of Binary Tree](https://leetcode.com/problems/maximum-depth-of-binary-tree/) — ★★★ · Easy

**Problem.** Number of nodes on the longest root-to-leaf path.  
**Signal.** Node's answer = 1 + best child answer → bottom-up recursion.  
*New here: The base bottom-up recursion: an empty tree answers 0.*

**Fine print.** Ask before coding:
- Depth counted in nodes or edges? → Nodes, so a single node has depth 1.
- Empty tree? → 0.
- Could the tree be very deep (10⁴ nodes in a line)? → Then recursion can hit Python's limit, so mention BFS.

> 💡 **Key insight.** A tree's height is one more than the taller of its two subtrees.

**Brute force →** Enumerate every root-to-leaf path and take the longest → O(n · h). The recursion computes each height once.

**Steps**
1. Empty node → 0.
2. Recurse on both children.
3. Return `1 + max(left, right)`.

**Complexity**

- **Time:** O(n) — each node visited once.
- **Space:** O(h) — recursion depth (h = height; O(n) when skewed).

In [3]:
def max_depth(root: TreeNode | None) -> int:
    """Height of the tree in nodes."""
    # an empty tree has no nodes
    if not root:
        return 0
    # this node plus the taller subtree
    return 1 + max(max_depth(root.left), max_depth(root.right))


assert max_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 3
assert max_depth(build_tree([1, None, 2])) == 2
assert max_depth(None) == 0
print(max_depth(build_tree([3, 9, 20, None, None, 15, 7])))

3


### 1.1.2 [111. Minimum Depth of Binary Tree](https://leetcode.com/problems/minimum-depth-of-binary-tree/) — ★ · Easy

**Problem.** Number of nodes on the shortest root-to-*leaf* path.  
**Signal.** Like max depth, but a missing child is not a leaf → BFS stops at the first leaf.  
*New here: The leaf definition trap, and BFS as the better tool for "shortest".*

**Fine print.** Ask before coding:
- Must the path end at a leaf (no children)? → Yes.
- Empty tree? → 0.

> 💡 **Key insight.** BFS visits nodes in order of depth, so the first leaf it meets is the shallowest one.

**Brute force →** DFS over all leaves → O(n) always; BFS can stop early at the first leaf.

**Steps**
1. Queue `(root, 1)`.
2. Pop a node; if it has no children, return its depth.
3. Otherwise enqueue existing children with depth + 1.

**Complexity**

- **Time:** O(n) worst case, often much less.
- **Space:** O(width) — the queue.

⚠️ **Trap.** `1 + min(left, right)` is wrong when one child is missing: that side is not a leaf.

In [4]:
def min_depth(root: TreeNode | None) -> int:
    """Depth of the shallowest leaf."""
    if not root:
        return 0
    queue = deque([(root, 1)])
    while queue:
        node, depth = queue.popleft()
        # the first leaf reached by BFS is the shallowest
        if not node.left and not node.right:
            return depth
        for child in (node.left, node.right):
            if child:
                queue.append((child, depth + 1))
    return 0


assert min_depth(build_tree([3, 9, 20, None, None, 15, 7])) == 2
assert min_depth(build_tree([2, None, 3, None, 4, None, 5, None, 6])) == 5
print(min_depth(build_tree([3, 9, 20, None, None, 15, 7])))

2


### 1.1.3 [100. Same Tree](https://leetcode.com/problems/same-tree/) — ★★★ · Easy

**Problem.** Are two trees structurally identical with equal values?  
**Signal.** Compare roots, then recurse on both pairs of children.  
*New here: Recursing on *two* trees in parallel.*

**Fine print.** Ask before coding:
- Same shape and same values? → Yes, both.
- Can either tree be empty? → Yes.

> 💡 **Key insight.** Two trees are equal iff their roots match and both pairs of subtrees are equal.

**Brute force →** Serialise both trees and compare strings → O(n) but extra space. Parallel recursion stops at the first difference.

**Steps**
1. If either node is empty, they are equal only if both are.
2. Values must match.
3. Left subtrees and right subtrees must both be the same.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [5]:
def is_same_tree(p: TreeNode | None, q: TreeNode | None) -> bool:
    """True if trees are identical."""
    # at least one empty: equal only if both are empty
    if not p or not q:
        return p is q
    # roots match and both subtree pairs match
    return (
        p.val == q.val
        and is_same_tree(p.left, q.left)
        and is_same_tree(p.right, q.right)
    )


assert is_same_tree(build_tree([1, 2, 3]), build_tree([1, 2, 3]))
assert not is_same_tree(build_tree([1, 2]), build_tree([1, None, 2]))
print(is_same_tree(build_tree([1, 2, 1]), build_tree([1, 1, 2])))

False


### 1.1.4 [226. Invert Binary Tree](https://leetcode.com/problems/invert-binary-tree/) — ★★★ · Easy

**Problem.** Mirror the tree.  
**Signal.** Swap children at every node.  
*New here: A recursion that transforms the tree and returns it.*

**Fine print.** Ask before coding:
- Modify in place and return the root? → Yes.
- Empty tree? → Return `None`.

> 💡 **Key insight.** A mirrored tree is the root with its *mirrored* subtrees swapped.

**Brute force →** Build a new mirrored copy → O(n) extra space. Swapping in place reuses the nodes.

**Steps**
1. Empty → return it.
2. Set `left = invert(right)` and `right = invert(left)` simultaneously.
3. Return the node.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [6]:
def invert_tree(root: TreeNode | None) -> TreeNode | None:
    """Mirror image of the tree."""
    if root:
        # swap the (already mirrored) subtrees
        root.left, root.right = invert_tree(root.right), invert_tree(root.left)
    return root


assert tree_to_list(invert_tree(build_tree([4, 2, 7, 1, 3, 6, 9]))) == [
    4,
    7,
    2,
    9,
    6,
    3,
    1,
]
print(tree_to_list(invert_tree(build_tree([2, 1, 3]))))

[2, 3, 1]


### 1.1.5 [101. Symmetric Tree](https://leetcode.com/problems/symmetric-tree/) — ★★ · Easy

**Problem.** Is the tree a mirror of itself?  
**Signal.** Compare the left subtree with the mirrored right subtree.  
*New here: Parallel recursion with *crossed* children.*

**Fine print.** Ask before coding:
- Compare values and shape? → Yes.
- Recursive or iterative? → Either; the follow-up asks for both.

> 💡 **Key insight.** A tree is symmetric iff its left subtree mirrors its right subtree: outer children match outer, inner match inner.

**Brute force →** Invert a copy of the tree and test Same Tree → O(n) extra space. The crossed comparison needs none.

**Steps**
1. `mirror(a, b)`: both empty → true; one empty → false.
2. Values must match.
3. Check `mirror(a.left, b.right)` and `mirror(a.right, b.left)`.
4. Start with `mirror(root.left, root.right)`.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [7]:
def is_symmetric(root: TreeNode | None) -> bool:
    """True if the tree mirrors itself."""

    def mirror(a: TreeNode | None, b: TreeNode | None) -> bool:
        if not a or not b:
            return a is b
        # outer pair and inner pair must both mirror
        return a.val == b.val and mirror(a.left, b.right) and mirror(a.right, b.left)

    return mirror(root.left, root.right) if root else True


assert is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3]))
assert not is_symmetric(build_tree([1, 2, 2, None, 3, None, 3]))
print(is_symmetric(build_tree([1, 2, 2, 3, 4, 4, 3])))

True


### 1.1.6 [572. Subtree of Another Tree](https://leetcode.com/problems/subtree-of-another-tree/) — ★★★ · Easy

**Problem.** Does `root` contain a subtree identical to `sub_root`?  
**Signal.** At each node of `root`, test Same Tree.  
*New here: Composing two recursions (search + compare).*

**Fine print.** Ask before coding:
- Must the subtree include all descendants, down to the leaves? → Yes, a whole subtree, not a partial match.
- Duplicate values? → Yes, so a match can fail and be retried at another node.

> 💡 **Key insight.** A matching subtree must start at some node of `root`, so test Same Tree at every node.

**Brute force →** This *is* the simple approach: O(m · n). Serialising both trees with null markers and doing a substring search (KMP) gives O(m + n).

**Steps**
1. Empty `root` → false.
2. If Same Tree holds here, true.
3. Otherwise search the left subtree, then the right.

**Complexity**

- **Time:** O(m · n) worst case.
- **Space:** O(h).

In [8]:
def is_subtree(root: TreeNode | None, sub_root: TreeNode | None) -> bool:
    """True if sub_root appears as a subtree of root."""
    if not root:
        return False
    # match starting here, or somewhere below
    return (
        is_same_tree(root, sub_root)
        or is_subtree(root.left, sub_root)
        or is_subtree(root.right, sub_root)
    )


assert is_subtree(build_tree([3, 4, 5, 1, 2]), build_tree([4, 1, 2]))
assert not is_subtree(
    build_tree([3, 4, 5, 1, 2, None, None, None, None, 0]), build_tree([4, 1, 2])
)
print(is_subtree(build_tree([3, 4, 5, 1, 2]), build_tree([4, 1, 2])))

True


### 1.1.7 [543. Diameter of Binary Tree](https://leetcode.com/problems/diameter-of-binary-tree/) — ★★ · Easy

**Problem.** Longest path (in edges) between any two nodes.  
**Signal.** Return height upward, update a global best with `left + right` at each node.  
*New here: "Global best + local return": record one thing, return another.*

**Fine print.** Ask before coding:
- Count edges or nodes? → Edges.
- Must the path go through the root? → No.

> 💡 **Key insight.** Every path has a highest node; there it uses the heights of both subtrees (`left + right` edges), but a parent can only extend one side.

**Brute force →** Compute heights separately at every node → O(n²). Returning heights while recording the best → O(n).

**Steps**
1. `height(node)` returns the subtree height (0 for empty).
2. At each node, update `best = max(best, left + right)`.
3. Return `1 + max(left, right)` to the parent.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [9]:
def diameter_of_binary_tree(root: TreeNode | None) -> int:
    """Longest path length in edges."""
    best = 0

    def height(node: TreeNode | None) -> int:
        nonlocal best
        if not node:
            return 0
        left, right = height(node.left), height(node.right)
        # longest path that bends at this node
        best = max(best, left + right)
        # a parent can extend only one side
        return 1 + max(left, right)

    height(root)
    return best


assert diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])) == 3
assert diameter_of_binary_tree(build_tree([1, 2])) == 1
print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))

3


### 1.1.8 [110. Balanced Binary Tree](https://leetcode.com/problems/balanced-binary-tree/) — ★★ · Easy

**Problem.** Is the tree height-balanced (subtree heights differ by ≤ 1 everywhere)?  
**Signal.** Height recursion that also signals failure.  
*New here: Encoding failure inside the return value (−1 = unbalanced).*

**Fine print.** Ask before coding:
- Height-balanced at every node, not just the root? → Every node.
- Empty tree? → Balanced.

> 💡 **Key insight.** One bottom-up pass can return the height *or* a failure flag, so no height is computed twice.

**Brute force →** At every node compute both heights from scratch → O(n²). Returning −1 on failure → O(n).

**Steps**
1. Empty → height 0.
2. Get both child heights.
3. If either is −1 or they differ by more than 1, return −1.
4. Otherwise return `1 + max`.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [10]:
def is_balanced(root: TreeNode | None) -> bool:
    """True if every node's subtrees differ in height by at most 1."""

    def height(node: TreeNode | None) -> int:
        if not node:
            return 0
        left, right = height(node.left), height(node.right)
        # -1 propagates "unbalanced somewhere below"
        if left < 0 or right < 0 or abs(left - right) > 1:
            return -1
        return 1 + max(left, right)

    return height(root) >= 0


assert is_balanced(build_tree([3, 9, 20, None, None, 15, 7]))
assert not is_balanced(build_tree([1, 2, 2, 3, 3, None, None, 4, 4]))
print(is_balanced(build_tree([3, 9, 20, None, None, 15, 7])))

True


### 1.1.9 [124. Binary Tree Maximum Path Sum](https://leetcode.com/problems/binary-tree-maximum-path-sum/) — ★★★ · Hard

**Problem.** Maximum sum of any path (any start and end node, values may be negative).  
**Signal.** Diameter pattern with sums: return best downward chain, update global with both sides.  
*New here: The diameter pattern with values, plus cutting negative branches.*

**Fine print.** Ask before coding:
- Can values be negative? → Yes, so a child's gain may be dropped (take max with 0).
- At least one node in the path? → Yes.
- Must the path go through the root? → No.

> 💡 **Key insight.** At each node the best bending path is `val + left_gain + right_gain`, where a negative gain is replaced by 0 (just don't take that branch).

**Brute force →** Try every pair of endpoints → O(n²) or worse. One post-order pass → O(n).

**Steps**
1. `gain(node)` = best sum of a downward chain starting at `node`.
2. Clamp child gains at 0.
3. Update `best` with `val + left + right`.
4. Return `val + max(left, right)`.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

⚠️ **Trap.** Initialise `best` to −∞, not 0 — an all-negative tree has a negative answer.

In [11]:
def max_path_sum(root: TreeNode | None) -> int:
    """Largest path sum anywhere in the tree."""
    best = float("-inf")

    def gain(node: TreeNode | None) -> int:
        nonlocal best
        if not node:
            return 0
        # a negative branch is better left out
        left, right = max(0, gain(node.left)), max(0, gain(node.right))
        # best path bending at this node
        best = max(best, node.val + left + right)
        # only one branch can continue upward
        return node.val + max(left, right)

    gain(root)
    return best


assert max_path_sum(build_tree([1, 2, 3])) == 6
assert max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])) == 42
assert max_path_sum(build_tree([-3])) == -3
print(max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])))

42


### 1.1.10 [236. Lowest Common Ancestor of a Binary Tree](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree/) — ★★ · Medium

**Problem.** Lowest node that has both `p` and `q` as descendants (a node is its own descendant).  
**Signal.** Bottom-up: return whichever target was found in each subtree.  
*New here: Returning *a found node* (or None) instead of a number.*

**Fine print.** Ask before coding:
- Are both `p` and `q` guaranteed to be in the tree? → Yes; if not, an extra check is needed.
- Can a node be its own ancestor? → Yes.
- Unique values? → Yes.

> 💡 **Key insight.** The LCA is the first node where `p` and `q` show up in different subtrees — or the node is one of them itself.

**Brute force →** Record both root-to-node paths and find the last common node → O(n) time but O(h) paths and two passes. One recursion does it directly.

**Steps**
1. Return the node if it is empty, `p` or `q`.
2. Recurse into both children.
3. Both sides non-empty → this node is the LCA.
4. Otherwise pass up whichever side is non-empty.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [12]:
def lowest_common_ancestor(root, p: TreeNode, q: TreeNode) -> TreeNode | None:
    """LCA of p and q in a binary tree."""
    # found a target (or nothing): report it upward
    if not root or root is p or root is q:
        return root
    left = lowest_common_ancestor(root.left, p, q)
    right = lowest_common_ancestor(root.right, p, q)
    # targets on both sides: this node splits them
    return root if left and right else left or right


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert lowest_common_ancestor(t, find(t, 5), find(t, 1)).val == 3
assert lowest_common_ancestor(t, find(t, 5), find(t, 4)).val == 5
print(lowest_common_ancestor(t, find(t, 6), find(t, 4)).val)

5


### 1.1.11 [1650. Lowest Common Ancestor of a Binary Tree III](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree-iii/) — ★ · Medium · Premium

**Problem.** Nodes have `parent` pointers (no root given); find the LCA of `p` and `q`.  
**Signal.** Two paths to the root that merge → same trick as Intersection of Two Linked Lists.  
*New here: Seeing a tree with parent pointers as two linked lists that merge.*

**Fine print.** Ask before coding:
- Do nodes have parent pointers and no root is given? → Yes.
- Is O(1) extra space possible? → Yes, with the two-pointer switch trick (like list intersection).

> 💡 **Key insight.** The paths from `p` and `q` up to the root are two linked lists sharing a tail; their first common node is the LCA.

**Brute force →** Store `p`'s ancestors in a set and walk up from `q` → O(h) space. Switching starts → O(1) space.

**Steps**
1. `a` walks up from `p`, `b` from `q`.
2. When one runs out of parents, restart it at the *other* starting node.
3. They meet at the LCA after equal total distances.

**Complexity**

- **Time:** O(h).
- **Space:** O(1).

In [13]:
def lca_with_parent(p, q):
    """LCA using parent pointers only."""
    a, b = p, q
    # equal total walk length makes them meet at the merge point
    while a is not b:
        a = a.parent if a.parent else q
        b = b.parent if b.parent else p
    return a


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
stack = [(t, None)]
while stack:
    node, parent = stack.pop()
    node.parent = parent
    stack += [(c, node) for c in (node.left, node.right) if c]
assert lca_with_parent(find(t, 5), find(t, 1)).val == 3
assert lca_with_parent(find(t, 5), find(t, 4)).val == 5
print(lca_with_parent(find(t, 7), find(t, 8)).val)

3


### 1.1.12 [979. Distribute Coins in Binary Tree](https://leetcode.com/problems/distribute-coins-in-binary-tree/) — ★ · Medium

**Problem.** `n` nodes hold `n` coins in total; a move passes one coin along an edge. Moves needed so every node has exactly one coin?  
**Signal.** Each edge carries the surplus/deficit of the subtree below it.  
*New here: Counting flow across edges with signed subtree totals.*

**Fine print.** Ask before coding:
- Total coins equal to the number of nodes? → Yes.
- Does one move pass one coin along one edge? → Yes.

> 💡 **Key insight.** The edge above a subtree must carry exactly `|coins in subtree − nodes in subtree|` coins, no matter how they are routed.

**Brute force →** Simulate moving coins step by step → hard to make optimal. Summing edge flows is exact and O(n).

**Steps**
1. `excess(node) = coins + excess(left) + excess(right) − 1`.
2. At each node, add `|excess(left)| + |excess(right)|` to the move count.
3. Return the node's excess to its parent.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [14]:
def distribute_coins(root: TreeNode | None) -> int:
    """Moves to give every node one coin."""
    moves = 0

    def excess(node: TreeNode | None) -> int:
        nonlocal moves
        if not node:
            return 0
        left, right = excess(node.left), excess(node.right)
        # coins that must cross each child edge (either direction)
        moves += abs(left) + abs(right)
        # surplus (+) or deficit (-) this subtree sends to its parent
        return node.val + left + right - 1

    excess(root)
    return moves


assert distribute_coins(build_tree([3, 0, 0])) == 2
assert distribute_coins(build_tree([0, 3, 0])) == 3
print(distribute_coins(build_tree([3, 0, 0])))

2


### 1.1.13 [337. House Robber III](https://leetcode.com/problems/house-robber-iii/) — ★ · Medium

**Problem.** Houses form a binary tree; robbing two directly linked houses is forbidden. Max loot?  
**Signal.** Tree DP: each node returns two values — (best if robbed, best if skipped).  
*New here: Tree DP: returning a tuple of states.*

**Fine print.** Ask before coding:
- Are only parent and child adjacent? → Yes, siblings can both be robbed.
- Non-negative values? → Yes.

> 💡 **Key insight.** A parent's choice only depends on whether each child was robbed, so each subtree returns `(best if robbed, best if skipped)`.

**Brute force →** Try rob/skip at every node → O(2ⁿ). Memoising on node also works; the tuple return is O(n) with no memo.

**Steps**
1. Empty → `(0, 0)`.
2. Robbed here → `val + left.skipped + right.skipped`.
3. Skipped here → `max(left) + max(right)`.
4. Answer = `max` of the root's pair.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [15]:
def rob_tree(root: TreeNode | None) -> int:
    """Maximum loot with no two adjacent nodes robbed."""

    def dfs(node: TreeNode | None) -> tuple[int, int]:
        """(best if this node is robbed, best if skipped)."""
        if not node:
            return 0, 0
        left, right = dfs(node.left), dfs(node.right)
        # robbing here forces both children to be skipped
        robbed = node.val + left[1] + right[1]
        # skipping here lets each child choose freely
        skipped = max(left) + max(right)
        return robbed, skipped

    return max(dfs(root))


assert rob_tree(build_tree([3, 2, 3, None, 3, None, 1])) == 7
assert rob_tree(build_tree([3, 4, 5, 1, 3, None, 1])) == 9
print(rob_tree(build_tree([3, 4, 5, 1, 3, None, 1])))

9


### 1.1.14 [968. Binary Tree Cameras](https://leetcode.com/problems/binary-tree-cameras/) — ★ · Hard

**Problem.** A camera watches its node, parent and children. Minimum cameras to watch every node?  
**Signal.** Greedy bottom-up with three states per node.  
*New here: Greedy decisions made bottom-up with node states.*

**Fine print.** Ask before coding:
- Does a camera cover its parent, itself and its children? → Yes.
- Are all node values 0? → Yes, values are irrelevant.

> 💡 **Key insight.** A leaf never needs a camera — its parent covers more — so place cameras as late (high) as possible: only when a child would otherwise stay uncovered.

**Brute force →** Tree DP over three states per node → O(n) but more cases. Greedy states → O(n) and short.

**Steps**
1. States: 0 = uncovered, 1 = has camera, 2 = covered; empty counts as covered.
2. Any child uncovered → camera here (state 1).
3. Else any child has a camera → covered (state 2).
4. Else uncovered (state 0); at the root, an uncovered state needs one more camera.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [16]:
def min_camera_cover(root: TreeNode | None) -> int:
    """Fewest cameras covering every node."""
    cameras = 0

    def state(node: TreeNode | None) -> int:
        """0 = uncovered, 1 = has camera, 2 = covered."""
        nonlocal cameras
        # missing children never need covering
        if not node:
            return 2
        left, right = state(node.left), state(node.right)
        # a child would stay uncovered: camera here
        if left == 0 or right == 0:
            cameras += 1
            return 1
        # a child's camera covers this node
        if left == 1 or right == 1:
            return 2
        # leave it for the parent to cover
        return 0

    # the root has no parent to rely on
    if state(root) == 0:
        cameras += 1
    return cameras


assert min_camera_cover(build_tree([0, 0, None, 0, 0])) == 1
assert min_camera_cover(build_tree([0, 0, None, 0, None, 0, None, None, 0])) == 2
assert min_camera_cover(build_tree([0])) == 1
print(min_camera_cover(build_tree([0, 0, None, 0, 0])))

1


## 1.2 Top-down recursion and paths

Top-down: the parent hands information *down* (remaining sum, number so far, max on the
path, direction of the last move). Check leaves explicitly — a leaf has no children at all.
When collecting whole paths, append before recursing and pop after (backtracking).

### 1.2.1 [112. Path Sum](https://leetcode.com/problems/path-sum/) — ★★ · Easy

**Problem.** Is there a root-to-leaf path summing to `target`?  
**Signal.** State flows downward: remaining sum.  
*New here: Passing state down instead of returning it up.*

**Fine print.** Ask before coding:
- Root to leaf, not any path? → Root to leaf.
- Negative values? → Yes, so you cannot stop early when the sum passes `target`.
- Empty tree? → `False`, even if `target` is 0.

> 💡 **Key insight.** Subtracting each node's value on the way down turns "path sums to target" into "remaining is 0 at a leaf".

**Brute force →** List every root-to-leaf path and sum it → O(n · h). Carrying the remainder → O(n).

**Steps**
1. Empty node → `False`.
2. Subtract the node's value from the target.
3. At a leaf, return whether the remainder is 0.
4. Otherwise try the left or right child.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [17]:
def has_path_sum(root: TreeNode | None, target: int) -> bool:
    """True if some root-to-leaf path sums to target."""
    if not root:
        return False
    # what the rest of the path must add up to
    target -= root.val
    # a leaf ends the path
    if not root.left and not root.right:
        return target == 0
    return has_path_sum(root.left, target) or has_path_sum(root.right, target)


t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
assert has_path_sum(t, 22)
assert not has_path_sum(build_tree([1, 2, 3]), 5)
print(has_path_sum(t, 22))

True


### 1.2.2 [113. Path Sum II](https://leetcode.com/problems/path-sum-ii/) — ★★ · Medium

**Problem.** All root-to-leaf paths summing to `target`.  
**Signal.** Top-down DFS with a path list and backtracking.  
*New here: A shared path list with append/pop (backtracking on a tree).*

**Fine print.** Ask before coding:
- Return node values, not nodes? → Values.
- Negative values? → Yes.

> 💡 **Key insight.** One list can represent the current root-to-node path if every append is undone by a pop when leaving the node.

**Brute force →** Pass a fresh copy of the path to every call → O(n · h) copies even for non-matching paths. Backtracking copies only matching paths.

**Steps**
1. Append the node's value; subtract it from the remainder.
2. At a leaf with remainder 0, save a *copy* of the path.
3. Recurse into both children.
4. Pop the node's value before returning.

**Complexity**

- **Time:** O(n · h) worst case — copying each matching path.
- **Space:** O(h) — the path and the recursion.

In [18]:
def path_sum_ii(root: TreeNode | None, target: int) -> list[list[int]]:
    """Every root-to-leaf path with the given sum."""
    out: list[list[int]] = []
    path: list[int] = []

    def dfs(node: TreeNode | None, remaining: int) -> None:
        if not node:
            return
        # enter the node
        path.append(node.val)
        remaining -= node.val
        # matching leaf: save a snapshot of the path
        if not node.left and not node.right and remaining == 0:
            out.append(path.copy())
        dfs(node.left, remaining)
        dfs(node.right, remaining)
        # leave the node
        path.pop()

    dfs(root, target)
    return out


t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
assert path_sum_ii(t, 22) == [[5, 4, 11, 2], [5, 8, 4, 5]]
print(path_sum_ii(t, 22))

[[5, 4, 11, 2], [5, 8, 4, 5]]


### 1.2.3 [437. Path Sum III](https://leetcode.com/problems/path-sum-iii/) — ★★ · Medium

**Problem.** Count downward paths (any start, any end) with sum `target`.  
**Signal.** Subarray Sum Equals K on every root-to-node path → prefix sums in a hash map.  
*New here: Prefix sums + hash map on a tree, with backtracking on the map.*

**Fine print.** Ask before coding:
- Must paths go downward? → Yes, parent to child only.
- Can a path start and end at any node? → Yes.
- Can sums overflow 32 bits? → Yes on LeetCode, so use 64-bit in other languages.

> 💡 **Key insight.** Any downward path is a contiguous piece of some root-to-node path, so the array trick "count earlier prefixes equal to `prefix − target`" works if the map holds only the current path's prefixes.

**Brute force →** Start a path-sum DFS from every node → O(n · h), O(n²) when skewed. Prefix map → O(n).

**Steps**
1. Seed `count = {0: 1}`.
2. At each node, add its value to the running prefix.
3. Paths ending here = `count[prefix − target]`.
4. Add `prefix` to the map, recurse, then remove it again.

**Complexity**

- **Time:** O(n).
- **Space:** O(h) — the map holds one prefix per depth.

In [19]:
def path_sum_iii(root: TreeNode | None, target: int) -> int:
    """Number of downward paths summing to target."""
    # prefix sums on the current root-to-node path
    count = Counter({0: 1})

    def dfs(node: TreeNode | None, prefix: int) -> int:
        if not node:
            return 0
        prefix += node.val
        # paths ending here that start lower on the current path
        found = count[prefix - target]
        count[prefix] += 1
        found += dfs(node.left, prefix) + dfs(node.right, prefix)
        # leaving this node: its prefix is no longer on the path
        count[prefix] -= 1
        return found

    return dfs(root, 0)


assert path_sum_iii(build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1]), 8) == 3
t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
assert path_sum_iii(t, 22) == 3
print(path_sum_iii(t, 22))

3


### 1.2.4 [129. Sum Root to Leaf Numbers](https://leetcode.com/problems/sum-root-to-leaf-numbers/) — ★★ · Medium

**Problem.** Each root-to-leaf path forms a number (digits); return the total.  
**Signal.** Top-down: pass the number built so far.  
*New here: Building a value digit by digit while descending.*

**Fine print.** Ask before coding:
- Are node values single digits 0–9? → Yes.
- Does the answer fit in 32 bits? → Yes, guaranteed.

> 💡 **Key insight.** Moving one level down appends a digit: `value * 10 + node.val`.

**Brute force →** Collect each path as a string and convert → O(n · h). Passing the number down → O(n).

**Steps**
1. `value = value * 10 + node.val`.
2. At a leaf, return `value`.
3. Otherwise return the sum of both children's results.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [20]:
def sum_numbers(root: TreeNode | None) -> int:
    """Sum of numbers formed by root-to-leaf paths."""

    def dfs(node: TreeNode | None, value: int) -> int:
        if not node:
            return 0
        # append this digit
        value = value * 10 + node.val
        if not node.left and not node.right:
            return value
        return dfs(node.left, value) + dfs(node.right, value)

    return dfs(root, 0)


assert sum_numbers(build_tree([1, 2, 3])) == 25
assert sum_numbers(build_tree([4, 9, 0, 5, 1])) == 1026
print(sum_numbers(build_tree([4, 9, 0, 5, 1])))

1026


### 1.2.5 [1448. Count Good Nodes in Binary Tree](https://leetcode.com/problems/count-good-nodes-in-binary-tree/) — ★★ · Medium

**Problem.** A node is good if no node on the path from the root to it has a greater value. Count them.  
**Signal.** Top-down: pass the max seen so far.  
*New here: Passing a running maximum down the path.*

**Fine print.** Ask before coding:
- Is the root always good? → Yes.
- Is equal allowed? → Yes, a node is good if it is `>=` every ancestor.

> 💡 **Key insight.** A node is good iff its value is at least the maximum on the path above it.

**Brute force →** For each node, walk its ancestors → O(n · h). Passing the path max → O(n).

**Steps**
1. Start with `path_max = root.val`.
2. Count the node if `val >= path_max`.
3. Recurse with `max(path_max, val)`.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [21]:
def good_nodes(root: TreeNode | None) -> int:
    """Number of nodes not smaller than any ancestor."""

    def dfs(node: TreeNode | None, path_max: int) -> int:
        if not node:
            return 0
        # nothing above is larger
        good = node.val >= path_max
        path_max = max(path_max, node.val)
        return good + dfs(node.left, path_max) + dfs(node.right, path_max)

    return dfs(root, root.val)


assert good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])) == 4
assert good_nodes(build_tree([3, 3, None, 4, 2])) == 3
print(good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])))

4


### 1.2.6 [257. Binary Tree Paths](https://leetcode.com/problems/binary-tree-paths/) — ★ · Easy

**Problem.** All root-to-leaf paths as strings `"1->2->5"`.  
**Signal.** Top-down DFS carrying the path.  
*New here: Passing an immutable path (a string), so no explicit pop is needed.*

**Fine print.** Ask before coding:
- Any output order? → Yes.
- Use the `"->"` separator? → Yes.

> 💡 **Key insight.** Strings are immutable, so each call gets its own path and nothing needs undoing.

**Brute force →** Same idea with a list + join at each leaf; the string version is shortest.

**Steps**
1. Append the node's value to the path string.
2. At a leaf, record the path.
3. Recurse with `path + "->"`.

**Complexity**

- **Time:** O(n · h) — building strings.
- **Space:** O(n · h) for the output.

In [22]:
def binary_tree_paths(root: TreeNode | None) -> list[str]:
    """Root-to-leaf paths as arrow strings."""
    out: list[str] = []

    def dfs(node: TreeNode | None, path: str) -> None:
        if not node:
            return
        path += str(node.val)
        if not node.left and not node.right:
            out.append(path)
        # each call gets its own string: no undo needed
        dfs(node.left, path + "->")
        dfs(node.right, path + "->")

    dfs(root, "")
    return out


assert binary_tree_paths(build_tree([1, 2, 3, None, 5])) == ["1->2->5", "1->3"]
print(binary_tree_paths(build_tree([1, 2, 3, None, 5])))

['1->2->5', '1->3']


### 1.2.7 [1372. Longest ZigZag Path in a Binary Tree](https://leetcode.com/problems/longest-zigzag-path-in-a-binary-tree/) — ★ · Medium

**Problem.** Longest path that alternates left/right child moves (counted in edges).  
**Signal.** Top-down state: (direction of last move, current length).  
*New here: State with two fields (direction, length) and a restart rule.*

**Fine print.** Ask before coding:
- Count edges? → Yes, a single node is 0.
- Can the path start at any node? → Yes.

> 💡 **Key insight.** Moving in the opposite direction to the last move extends the zigzag; moving the same way starts a new zigzag of length 1.

**Brute force →** Start a zigzag walk from every node in both directions → O(n · h). Carrying state → O(n).

**Steps**
1. Start at both children of the root with length 1.
2. Going left after a right move → `length + 1`; going left after a left move → 1.
3. Symmetric for right.
4. Track the maximum length.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [23]:
def longest_zig_zag(root: TreeNode | None) -> int:
    """Longest alternating downward path length."""
    best = 0

    def dfs(node: TreeNode | None, went_left: bool, length: int) -> None:
        nonlocal best
        if not node:
            return
        best = max(best, length)
        # going left continues the zigzag only if we just went right
        dfs(node.left, True, 1 if went_left else length + 1)
        # going right continues it only if we just went left
        dfs(node.right, False, length + 1 if went_left else 1)

    dfs(root.left, True, 1)
    dfs(root.right, False, 1)
    return best


t = build_tree([1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1])
assert longest_zig_zag(t) == 3
assert longest_zig_zag(build_tree([1, 1, 1, None, 1, None, None, 1, 1, None, 1])) == 4
assert longest_zig_zag(build_tree([1])) == 0
print(longest_zig_zag(t))

3


### 1.2.8 [872. Leaf-Similar Trees](https://leetcode.com/problems/leaf-similar-trees/) — ★ · Easy

**Problem.** Do two trees have the same leaf value sequence (left to right)?  
**Signal.** Collect leaves with DFS and compare.  
*New here: A recursive *generator* (`yield from`) to stream values.*

**Fine print.** Ask before coding:
- Leaves in left-to-right order? → Yes.
- Can the trees have different shapes? → Yes, only the leaf sequences matter.

> 💡 **Key insight.** A preorder DFS meets leaves in left-to-right order, so the two leaf streams can be compared directly.

**Brute force →** Collect both lists and compare → O(n) space. Comparing generators lazily can stop at the first difference.

**Steps**
1. `leaves(node)` yields the node's value if it is a leaf.
2. Then yields from the left and right subtrees.
3. Compare the two sequences.

**Complexity**

- **Time:** O(n + m).
- **Space:** O(h) for the generators (plus the lists here).

In [24]:
def leaf_similar(a: TreeNode | None, b: TreeNode | None) -> bool:
    """True if both trees have the same leaf sequence."""

    def leaves(node: TreeNode | None):
        if node:
            if not node.left and not node.right:
                yield node.val
            # left leaves come before right leaves
            yield from leaves(node.left)
            yield from leaves(node.right)

    return list(leaves(a)) == list(leaves(b))


a = build_tree([3, 5, 1, 6, 2, 9, 8, None, None, 7, 4])
b = build_tree([3, 5, 1, 6, 7, 4, 2, None, None, None, None, None, None, 9, 8])
assert leaf_similar(a, b)
assert not leaf_similar(build_tree([1, 2, 3]), build_tree([1, 3, 2]))
print(leaf_similar(a, b))

True


## 1.3 Construction, transformation and serialisation

Building or reshaping a tree. Key facts: preorder starts with the root, postorder ends with
it, inorder puts the root between its subtrees. With a `value → inorder index` map, each
subtree is an index range.

### 1.3.1 [105. Construct Binary Tree from Preorder and Inorder Traversal](https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/) — ★★★ · Medium

**Problem.** Rebuild the tree from its preorder and inorder sequences (values distinct).  
**Signal.** Preorder gives the root; inorder splits left and right subtrees.  
*New here: Rebuilding a tree recursively from index ranges.*

**Fine print.** Ask before coding:
- Unique values? → Yes; with duplicates the tree is ambiguous.
- Are the two traversals guaranteed consistent? → Yes.

> 💡 **Key insight.** The next preorder value is the root of the current range; its position in inorder splits the range into the left and right subtrees.

**Brute force →** Search for the root in inorder with `.index()` each time and slice lists → O(n²). An index map and range bounds → O(n).

**Steps**
1. Map each value to its inorder index.
2. `make(lo, hi)`: empty range → `None`.
3. Take the next preorder value as the root; find its inorder index `m`.
4. Build left from `(lo, m − 1)` then right from `(m + 1, hi)` — order matters, preorder is consumed left first.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) — the map and recursion.

In [25]:
def build_from_pre_in(preorder: list[int], inorder: list[int]) -> TreeNode | None:
    """Tree with the given preorder and inorder traversals."""
    # value -> position in inorder
    index = {v: i for i, v in enumerate(inorder)}
    # preorder is consumed root-first, left subtree before right
    pre = iter(preorder)

    def make(lo: int, hi: int) -> TreeNode | None:
        """Subtree covering inorder[lo..hi]."""
        if lo > hi:
            return None
        root = TreeNode(next(pre))
        m = index[root.val]
        # inorder left of m is the left subtree, right of m the right subtree
        root.left = make(lo, m - 1)
        root.right = make(m + 1, hi)
        return root

    return make(0, len(inorder) - 1)


t = build_from_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
assert tree_to_list(t) == [3, 9, 20, None, None, 15, 7]
print(tree_to_list(t))

[3, 9, 20, None, None, 15, 7]


### 1.3.2 [106. Construct Binary Tree from Inorder and Postorder Traversal](https://leetcode.com/problems/construct-binary-tree-from-inorder-and-postorder-traversal/) — ★★ · Medium

**Problem.** Rebuild the tree from inorder and postorder sequences.  
**Signal.** Postorder read backwards is root → right → left.  
*New here: The mirror construction: consume from the end, build right first.*

**Fine print.** Ask before coding:
- Unique values? → Yes.
- Build the right subtree first when consuming postorder from the end? → Yes, that is the trap.

> 💡 **Key insight.** Reversed postorder visits root, then right subtree, then left — so build the right subtree first.

**Brute force →** Same O(n²) slicing version as above; the index map gives O(n).

**Steps**
1. Map values to inorder indices.
2. Read postorder backwards.
3. Each value is the root of the current range.
4. Build the right range first, then the left.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [26]:
def build_from_in_post(inorder: list[int], postorder: list[int]) -> TreeNode | None:
    """Tree with the given inorder and postorder traversals."""
    index = {v: i for i, v in enumerate(inorder)}
    # reversed postorder: root, right subtree, left subtree
    post = reversed(postorder)

    def make(lo: int, hi: int) -> TreeNode | None:
        if lo > hi:
            return None
        root = TreeNode(next(post))
        m = index[root.val]
        # right first, to match the consumption order
        root.right = make(m + 1, hi)
        root.left = make(lo, m - 1)
        return root

    return make(0, len(inorder) - 1)


t = build_from_in_post([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])
assert tree_to_list(t) == [3, 9, 20, None, None, 15, 7]
print(tree_to_list(t))

[3, 9, 20, None, None, 15, 7]


### 1.3.3 [114. Flatten Binary Tree to Linked List](https://leetcode.com/problems/flatten-binary-tree-to-linked-list/) — ★★ · Medium

**Problem.** Flatten in place into a right-skewed "list" in preorder.  
**Signal.** Morris-style: splice each left subtree between the node and its right subtree.  
*New here: O(1)-space rewiring of a tree (Morris-style).*

**Fine print.** Ask before coding:
- In place, using the `right` pointer, with `left` set to `None`? → Yes.
- O(1) extra space? → The follow-up asks for it, which leads to the Morris-style approach.

> 💡 **Key insight.** In preorder, a node's right subtree comes right after the *last* node of its left subtree, so hang the right subtree there and move the left subtree to the right.

**Brute force →** Collect preorder nodes in a list and relink → O(n) space. Splicing → O(1).

**Steps**
1. At `cur` with a left child, find the rightmost node of the left subtree.
2. Attach `cur.right` to that node.
3. Move the left subtree to `cur.right`; clear `cur.left`.
4. Step to `cur.right` and repeat.

**Complexity**

- **Time:** O(n) — each node is on a "rightmost" walk at most once.
- **Space:** O(1).

In [27]:
def flatten(root: TreeNode | None) -> None:
    """Flatten tree to a preorder right-chain in place."""
    cur = root
    while cur:
        if cur.left:
            # last node of the left subtree in preorder
            tail = cur.left
            while tail.right:
                tail = tail.right
            # right subtree follows it
            tail.right = cur.right
            # left subtree moves to the right
            cur.right, cur.left = cur.left, None
        cur = cur.right


t = build_tree([1, 2, 5, 3, 4, None, 6])
flatten(t)
assert tree_to_list(t) == [1, None, 2, None, 3, None, 4, None, 5, None, 6]
print(tree_to_list(t))

[1, None, 2, None, 3, None, 4, None, 5, None, 6]


### 1.3.4 [617. Merge Two Binary Trees](https://leetcode.com/problems/merge-two-binary-trees/) — ★ · Easy

**Problem.** Overlay two trees: overlapping nodes sum, otherwise take the existing node.  
**Signal.** Simultaneous recursion on both trees.  
*New here: Building a new tree from a parallel recursion.*

**Fine print.** Ask before coding:
- Create a new tree or reuse nodes? → Either; reusing nodes is fine.
- Can either root be `None`? → Yes.

> 💡 **Key insight.** Where only one tree has a node, its whole subtree can be reused as is.

**Brute force →** Fill both trees into full arrays and add → wasteful. Parallel recursion stops wherever one side is empty.

**Steps**
1. If either node is empty, return the other.
2. Otherwise create a node with the summed value.
3. Merge left with left, right with right.

**Complexity**

- **Time:** O(min(m, n)) — only overlapping nodes are visited.
- **Space:** O(min height).

In [28]:
def merge_trees(a: TreeNode | None, b: TreeNode | None) -> TreeNode | None:
    """Overlay of two trees."""
    # one side missing: reuse the other subtree whole
    if not a or not b:
        return a or b
    return TreeNode(
        a.val + b.val, merge_trees(a.left, b.left), merge_trees(a.right, b.right)
    )


t = merge_trees(build_tree([1, 3, 2, 5]), build_tree([2, 1, 3, None, 4, None, 7]))
assert tree_to_list(t) == [3, 4, 5, 5, 4, None, 7]
print(tree_to_list(t))

[3, 4, 5, 5, 4, None, 7]


### 1.3.5 [297. Serialize and Deserialize Binary Tree](https://leetcode.com/problems/serialize-and-deserialize-binary-tree/) — ★★★ · Hard

**Problem.** Encode a tree to a string and decode it back.  
**Signal.** Preorder with explicit null markers is unambiguous.  
*New here: Null markers make a single traversal enough to rebuild a tree.*

**Fine print.** Ask before coding:
- Any format is fine as long as it round-trips? → Yes.
- Negative or multi-digit values? → Yes, so use a delimiter.
- Stateless? → Yes, no globals.

> 💡 **Key insight.** With a marker for every missing child, preorder alone pins down the shape: the decoder reads tokens in exactly the order the encoder wrote them.

**Brute force →** Store preorder *and* inorder → needs distinct values and two lists. Preorder with `#` works for any values.

**Steps**
1. Serialise: preorder, writing `#` for `None`, joined by commas.
2. Deserialise: an iterator over tokens.
3. `#` → `None`; otherwise create the node and build its left, then right subtree.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [29]:
def serialize(root: TreeNode | None) -> str:
    """Preorder string with # for null."""
    out: list[str] = []

    def dfs(node: TreeNode | None) -> None:
        # explicit marker for every missing child
        if not node:
            out.append("#")
            return
        out.append(str(node.val))
        dfs(node.left)
        dfs(node.right)

    dfs(root)
    return ",".join(out)


def deserialize(data: str) -> TreeNode | None:
    """Inverse of serialize."""
    tokens = iter(data.split(","))

    def make() -> TreeNode | None:
        # read tokens in the same preorder they were written
        tok = next(tokens)
        if tok == "#":
            return None
        return TreeNode(int(tok), make(), make())

    return make()


t = build_tree([1, 2, 3, None, None, 4, 5])
assert tree_to_list(deserialize(serialize(t))) == [1, 2, 3, None, None, 4, 5]
assert deserialize(serialize(None)) is None
print(serialize(t))

1,2,#,#,3,4,#,#,5,#,#


### 1.3.6 [863. All Nodes Distance K in Binary Tree](https://leetcode.com/problems/all-nodes-distance-k-in-binary-tree/) — ★★ · Medium

**Problem.** Values of all nodes at distance `k` from a `target` node.  
**Signal.** Distance can go upward → turn the tree into an undirected graph, then BFS.  
*New here: Converting a tree into a graph (adding parent edges).*

**Fine print.** Ask before coding:
- Unique values? → Yes.
- Is `target` a node, not a value? → A node.
- Any output order? → Yes.

> 💡 **Key insight.** Distance can go up through parents, so once every node knows its parent, it is a plain BFS on an undirected graph.

**Brute force →** For every node, compute its distance to the target via LCA → O(n · h). Parent map + BFS → O(n).

**Steps**
1. DFS to record each node's parent.
2. BFS from `target` over left, right and parent edges.
3. Keep a visited set (the graph now has back edges).
4. After `k` levels, the frontier is the answer.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [30]:
def distance_k(root: TreeNode, target: TreeNode, k: int) -> list[int]:
    """Values at exactly k edges from target."""
    # parent pointers turn the tree into an undirected graph
    parent: dict[TreeNode, TreeNode | None] = {root: None}
    stack = [root]
    while stack:
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                stack.append(child)
    # BFS k levels out from target
    frontier, seen = [target], {target}
    for _ in range(k):
        nxt = []
        for node in frontier:
            for nb in (node.left, node.right, parent[node]):
                if nb and nb not in seen:
                    seen.add(nb)
                    nxt.append(nb)
        frontier = nxt
    return sorted(node.val for node in frontier)


t = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert distance_k(t, find(t, 5), 2) == [1, 4, 7]
single = build_tree([1])
assert distance_k(single, single, 3) == []
print(distance_k(t, find(t, 5), 2))

[1, 4, 7]


### 1.3.7 [222. Count Complete Tree Nodes](https://leetcode.com/problems/count-complete-tree-nodes/) — ★★ · Easy

**Problem.** Count nodes of a *complete* tree in less than O(n).  
**Signal.** If leftmost and rightmost depths match, the subtree is perfect: `2^h − 1` nodes.  
*New here: Using a structural guarantee to skip whole subtrees.*

**Fine print.** Ask before coding:
- Is the tree guaranteed complete? → Yes; that is what allows O(log² n).
- Faster than O(n)? → That is the whole point.

> 💡 **Key insight.** In a complete tree, equal leftmost and rightmost depths mean the subtree is perfect and its size is `2^h − 1`; otherwise one child is perfect and only the other needs recursion.

**Brute force →** Count every node → O(n). The perfect-subtree shortcut → O(log² n).

**Steps**
1. Measure depth along the far-left and far-right edges.
2. Equal → return `2^h − 1`.
3. Otherwise return `1 + count(left) + count(right)`.

**Complexity**

- **Time:** O(log² n) — O(log n) levels of recursion, each measuring O(log n) depths.
- **Space:** O(log n).

In [31]:
def count_nodes(root: TreeNode | None) -> int:
    """Node count of a complete binary tree."""
    if not root:
        return 0
    # depth along the leftmost and rightmost edges
    left_h = right_h = 0
    node = root
    while node:
        left_h, node = left_h + 1, node.left
    node = root
    while node:
        right_h, node = right_h + 1, node.right
    # equal edges: perfect tree, use the formula
    if left_h == right_h:
        return 2**left_h - 1
    return 1 + count_nodes(root.left) + count_nodes(root.right)


assert count_nodes(build_tree([1, 2, 3, 4, 5, 6])) == 6
assert count_nodes(build_tree(list(range(1, 16)))) == 15
assert count_nodes(None) == 0
print(count_nodes(build_tree([1, 2, 3, 4, 5, 6])))

6


---
# 2) Binary Tree BFS

**In plain English.** Visit the tree one level at a time, top to bottom, using a queue.
Snapshot the queue's size at the start of a level — exactly that many nodes belong to it.

> **Analogy.** Ripples on a pond: everything one step from the stone is touched first,
> then everything two steps away, and so on.

```python
queue = deque([root])
while queue:
    level = []
    for _ in range(len(queue)):        # exactly the nodes of the current level
        node = queue.popleft()
        level.append(node.val)
        queue.extend(c for c in (node.left, node.right) if c)
```

**Recognition signals**

- "level order", "each level", "row by row", "zigzag"
- "right side view", "leftmost / rightmost per level", "average / max per level"
- "minimum depth", "nearest", "shortest" (BFS reaches shallow nodes first)
- "width", "complete tree", "vertical / column order"

**Common bugs**

- Using `len(queue)` inside the loop instead of a snapshot taken before it.
- `list.pop(0)` instead of `deque.popleft()` → O(n) per pop.
- Pushing `None` children by accident (except when you need gaps, as in Completeness).
- Huge position indices in width problems — re-base each level.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Tree DFS | depth first (recursion / stack); BFS is breadth first (queue) |
| Graph BFS | same queue, but needs a `visited` set; tree BFS never revisits |
| Vertical order | still BFS, but grouped by column index instead of by level |

**Learning path.** Level order (the template) → per-level summaries (right view, zigzag,
averages, max sum) → positional indices (width) → gaps in the queue (completeness) → using
the level above as a linked list (next pointers, O(1) space) → column grouping (vertical
order, then with value tie-breaks).

## 2.1 Level-order traversal

Every question here is the level template plus one twist: what you record per level, what
extra data rides along with each node (an index, a column), or what you do with gaps.

### 2.1.1 [102. Binary Tree Level Order Traversal](https://leetcode.com/problems/binary-tree-level-order-traversal/) — ★★★ · Medium

**Problem.** Values level by level, left to right.  
**Signal.** Level by level → BFS with the level-size loop.  
*New here: The level BFS template.*

**Fine print.** Ask before coding:
- Return a list per level? → Yes, left to right.
- Empty tree? → `[]`.

> 💡 **Key insight.** At the start of each round, the queue holds exactly one full level, so its size tells you how many pops belong to that level.

**Brute force →** DFS recording `depth` and appending to `out[depth]` → also O(n); BFS gives the order naturally.

**Steps**
1. Queue the root.
2. Snapshot `len(queue)`; pop that many nodes, collecting their values.
3. Push each popped node's children.
4. Append the level and repeat.

**Complexity**

- **Time:** O(n).
- **Space:** O(width) — the queue holds at most one or two levels.

In [32]:
def level_order(root: TreeNode | None) -> list[list[int]]:
    """Node values grouped by level."""
    out: list[list[int]] = []
    queue = deque([root] if root else [])
    while queue:
        level = []
        # the queue holds exactly this level right now
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            # children form the next level
            queue.extend(c for c in (node.left, node.right) if c)
        out.append(level)
    return out


assert level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
assert level_order(None) == []
print(level_order(build_tree([3, 9, 20, None, None, 15, 7])))

[[3], [9, 20], [15, 7]]


### 2.1.2 [199. Binary Tree Right Side View](https://leetcode.com/problems/binary-tree-right-side-view/) — ★★ · Medium

**Problem.** Values visible from the right side (last node of each level).  
**Signal.** Level-order, keep the last value of each level.  
*New here: Reusing the template and picking one value per level.*

**Fine print.** Ask before coding:
- The rightmost node of each level, even if it comes from the left subtree? → Yes.

> 💡 **Key insight.** The node you see from the right is the last node of each level.

**Brute force →** Same O(n); the DFS alternative visits right first and records the first node at each new depth.

**Steps**
1. Run level order.
2. Take the last value of every level.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) here (the levels); O(width) if recorded on the fly.

In [33]:
def right_side_view(root: TreeNode | None) -> list[int]:
    """Rightmost value on each level."""
    # the last node of a level is the one visible from the right
    return [level[-1] for level in level_order(root)]


assert right_side_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert right_side_view(build_tree([1, 2, 3, 4, None, None, None, 5])) == [1, 3, 4, 5]
print(right_side_view(build_tree([1, 2, 3, None, 5, None, 4])))

[1, 3, 4]


### 2.1.3 [103. Binary Tree Zigzag Level Order Traversal](https://leetcode.com/problems/binary-tree-zigzag-level-order-traversal/) — ★★ · Medium

**Problem.** Level order, alternating left→right and right→left.  
**Signal.** Level BFS, reverse every other level.  
*New here: Post-processing levels by parity.*

**Fine print.** Ask before coding:
- Does the first level go left to right? → Yes.

> 💡 **Key insight.** The traversal stays the same; only the output order of odd levels flips.

**Brute force →** Alternate the order children are pushed — easy to get wrong. Reversing odd levels is simpler and still O(n).

**Steps**
1. Run level order.
2. Reverse levels with odd index.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [34]:
def zigzag_level_order(root: TreeNode | None) -> list[list[int]]:
    """Level order with alternating direction."""
    levels = level_order(root)
    # odd levels read right to left
    return [lvl if i % 2 == 0 else lvl[::-1] for i, lvl in enumerate(levels)]


assert zigzag_level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [
    [3],
    [20, 9],
    [15, 7],
]
print(zigzag_level_order(build_tree([1, 2, 3, 4, None, None, 5])))

[[1], [3, 2], [4, 5]]


### 2.1.4 [637. Average of Levels in Binary Tree](https://leetcode.com/problems/average-of-levels-in-binary-tree/) — ★★ · Easy

**Problem.** Average value of each level.  
**Signal.** Level BFS.  
*New here: A per-level aggregate.*

**Fine print.** Ask before coding:
- Float output? → Yes, within 1e-5.
- Can level sums overflow 32 bits? → Yes, use 64-bit in other languages.

> 💡 **Key insight.** Once nodes are grouped by level, the average is sum / count per group.

**Brute force →** DFS with per-depth sums and counts → same O(n).

**Steps**
1. Run level order.
2. Average each level.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [35]:
def average_of_levels(root: TreeNode | None) -> list[float]:
    """Mean value per level."""
    return [sum(lvl) / len(lvl) for lvl in level_order(root)]


assert average_of_levels(build_tree([3, 9, 20, None, None, 15, 7])) == [3.0, 14.5, 11.0]
print(average_of_levels(build_tree([3, 9, 20, None, None, 15, 7])))

[3.0, 14.5, 11.0]


### 2.1.5 [1161. Maximum Level Sum of a Binary Tree](https://leetcode.com/problems/maximum-level-sum-of-a-binary-tree/) — ★ · Medium

**Problem.** The smallest level number (root = 1) with the maximum sum.  
**Signal.** Level BFS.  
*New here: Tie-breaking by first occurrence.*

**Fine print.** Ask before coding:
- Is the root at level 1? → Yes.
- Tie on the maximum sum? → Return the smallest level.
- Negative values? → Yes, so start `best` at −∞.

> 💡 **Key insight.** `list.index(max(...))` returns the *first* level with the maximum, which is the required tie-break.

**Brute force →** Same O(n) with DFS per-depth sums.

**Steps**
1. Sum each level.
2. Return the 1-based index of the first maximum.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [36]:
def max_level_sum(root: TreeNode | None) -> int:
    """1-based level with the largest sum."""
    sums = [sum(lvl) for lvl in level_order(root)]
    # index() picks the smallest level on ties
    return sums.index(max(sums)) + 1


assert max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])) == 2
assert (
    max_level_sum(
        build_tree([989, None, 10250, 98693, -89388, None, None, None, -32127])
    )
    == 2
)
print(max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])))

2


### 2.1.6 [662. Maximum Width of Binary Tree](https://leetcode.com/problems/maximum-width-of-binary-tree/) — ★★ · Medium

**Problem.** Maximum width of any level, counting null gaps between the end nodes.  
**Signal.** Give nodes heap-style indices (`2i`, `2i + 1`); width = last − first + 1.  
*New here: Carrying a position index with each node.*

**Fine print.** Ask before coding:
- Count the null gaps between the end nodes? → Yes.
- Does the answer fit in 32 bits? → Yes, but intermediate indices do not, so rebase each level.

> 💡 **Key insight.** With heap numbering (children of `i` at `2i` and `2i + 1`), the width of a level is `last index − first index + 1`, gaps included.

**Brute force →** Push `None` placeholders for every gap → exponential in deep sparse trees. Indices avoid storing gaps.

**Steps**
1. Queue `(root, 0)`.
2. For each level, note the first index and re-base indices by subtracting it.
3. Children get `2i` and `2i + 1`.
4. Width = last re-based index + 1; keep the max.

**Complexity**

- **Time:** O(n).
- **Space:** O(width).

⚠️ **Trap.** Without re-basing, indices double each level and grow huge in skewed trees.

In [37]:
def width_of_binary_tree(root: TreeNode | None) -> int:
    """Largest level width including gaps."""
    best = 0
    queue = deque([(root, 0)])
    while queue:
        first = queue[0][1]
        last = first
        for _ in range(len(queue)):
            node, idx = queue.popleft()
            # re-base so indices stay small
            idx -= first
            last = idx
            # heap numbering for children
            if node.left:
                queue.append((node.left, 2 * idx))
            if node.right:
                queue.append((node.right, 2 * idx + 1))
        best = max(best, last + 1)
    return best


assert width_of_binary_tree(build_tree([1, 3, 2, 5, 3, None, 9])) == 4
assert width_of_binary_tree(build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])) == 7
assert width_of_binary_tree(build_tree([1, 3, 2, 5])) == 2
print(width_of_binary_tree(build_tree([1, 3, 2, 5, 3, None, 9])))

4


### 2.1.7 [958. Check Completeness of a Binary Tree](https://leetcode.com/problems/check-completeness-of-a-binary-tree/) — ★ · Medium

**Problem.** Is the tree complete (all levels full except the last, filled from the left)?  
**Signal.** BFS including `None`s: no node may appear after the first `None`.  
*New here: Deliberately queueing `None` to detect gaps.*

**Fine print.** Ask before coding:
- Must the last level be filled from the left? → Yes.
- Does a null followed by a node make it incomplete? → Yes.

> 💡 **Key insight.** In level order a complete tree has all its nodes first and all its gaps last, so a real node after a gap means "not complete".

**Brute force →** Count nodes and check heap indices are all ≤ n → also O(n), more code.

**Steps**
1. BFS, pushing both children even when `None`.
2. Popping a `None` sets `seen_gap`.
3. Popping a real node after a gap → `False`.

**Complexity**

- **Time:** O(n).
- **Space:** O(width).

In [38]:
def is_complete_tree(root: TreeNode | None) -> bool:
    """True if the tree is complete."""
    queue = deque([root])
    seen_gap = False
    while queue:
        node = queue.popleft()
        if not node:
            seen_gap = True
        else:
            # a node after a gap breaks completeness
            if seen_gap:
                return False
            # push gaps too, so they show up in order
            queue.extend([node.left, node.right])
    return True


assert is_complete_tree(build_tree([1, 2, 3, 4, 5, 6]))
assert not is_complete_tree(build_tree([1, 2, 3, 4, 5, None, 7]))
print(is_complete_tree(build_tree([1, 2, 3, 4, 5, 6])))

True


### 2.1.8 [117. Populating Next Right Pointers in Each Node II](https://leetcode.com/problems/populating-next-right-pointers-in-each-node-ii/) — ★★ · Medium

**Problem.** Set each node's `next` to its right neighbour on the same level, O(1) extra space.  
**Signal.** Use the already-linked level above as a linked list to wire the level below.  
*New here: Level traversal without a queue, by walking the previous level's `next` links.*

**Fine print.** Ask before coding:
- Is the tree not necessarily perfect? → Correct, so you cannot assume every node has two children.
- O(1) extra space? → Yes, walk each level through the `next` pointers you already set.

> 💡 **Key insight.** Once a level is linked by `next`, it is a linked list you can walk to visit its children in order and link *them* with a dummy head.

**Brute force →** Level BFS with a queue, linking consecutive nodes → O(width) space. Walking `next` pointers → O(1).

**Steps**
1. `level` starts at the root.
2. Walk the level via `next`, appending each child to a new chain (dummy + tail).
3. End the chain with `None`.
4. Move `level` to `dummy.next` and repeat.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [39]:
def connect(root: TreeNode | None) -> TreeNode | None:
    """Link each node to its right neighbour."""
    level = root
    while level:
        # build the child level as a chain
        dummy = tail = TreeNode()
        node = level
        # walk the current level through its next links
        while node:
            for child in (node.left, node.right):
                if child:
                    tail.next = tail = child
            node = getattr(node, "next", None)
        tail.next = None
        # descend to the first node of the child level
        level = dummy.next
    return root


t = connect(build_tree([1, 2, 3, 4, 5, None, 7]))
row, node = [], t.left.left
while node:
    row.append(node.val)
    node = node.next
assert row == [4, 5, 7] and t.left.next is t.right
print(row)

[4, 5, 7]


### 2.1.9 [314. Binary Tree Vertical Order Traversal](https://leetcode.com/problems/binary-tree-vertical-order-traversal/) — ★ · Medium · Premium

**Problem.** Group values by column (root at 0, left −1, right +1), top to bottom, left to right.  
**Signal.** BFS (for top-to-bottom order) carrying the column index.  
*New here: Grouping by a column key while keeping BFS order.*

**Fine print.** Ask before coding:
- Within a column, top to bottom, then left to right? → Yes, so use BFS, not DFS.
- Return columns from left to right? → Yes.

> 💡 **Key insight.** BFS already gives top-to-bottom, left-to-right order, so appending to `columns[col]` in BFS order is exactly the required order.

**Brute force →** DFS with `(row, col)` and a sort → O(n log n). BFS + column map → O(n) plus sorting the column keys.

**Steps**
1. BFS with `(node, col)`.
2. Append each value to `columns[col]`.
3. Left child gets `col − 1`, right child `col + 1`.
4. Output columns from the smallest key to the largest.

**Complexity**

- **Time:** O(n) (+ O(w log w) to sort w column keys).
- **Space:** O(n).

In [40]:
def vertical_order(root: TreeNode | None) -> list[list[int]]:
    """Columns of values, left to right."""
    if not root:
        return []
    columns: dict[int, list[int]] = defaultdict(list)
    queue = deque([(root, 0)])
    while queue:
        node, col = queue.popleft()
        # BFS order = top-to-bottom, left-to-right within a column
        columns[col].append(node.val)
        if node.left:
            queue.append((node.left, col - 1))
        if node.right:
            queue.append((node.right, col + 1))
    return [columns[c] for c in sorted(columns)]


assert vertical_order(build_tree([3, 9, 20, None, None, 15, 7])) == [
    [9],
    [3, 15],
    [20],
    [7],
]
assert vertical_order(build_tree([3, 9, 8, 4, 0, 1, 7])) == [
    [4],
    [9],
    [3, 0, 1],
    [8],
    [7],
]
print(vertical_order(build_tree([3, 9, 20, None, None, 15, 7])))

[[9], [3, 15], [20], [7]]


### 2.1.10 [987. Vertical Order Traversal of a Binary Tree](https://leetcode.com/problems/vertical-order-traversal-of-a-binary-tree/) — ★ · Hard

**Problem.** Same columns, but nodes sharing a row and column are ordered by value.  
**Signal.** Collect `(col, row, val)` triples and sort.  
*New here: Encoding a multi-key ordering as a tuple sort.*

**Fine print.** Ask before coding:
- Same row and column? → Sort by value.
- How does this differ from 314? → Ties are broken by value, not by BFS order.

> 💡 **Key insight.** The required order is lexicographic on `(column, row, value)`, which is exactly how Python sorts tuples.

**Brute force →** BFS with per-row sorting of collisions → fiddly. One tuple sort → O(n log n).

**Steps**
1. DFS recording `(col, row, val)` for every node.
2. Sort the triples.
3. Group values by column in sorted order.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

In [41]:
def vertical_traversal(root: TreeNode | None) -> list[list[int]]:
    """Vertical order with value tie-break."""
    cells: list[tuple[int, int, int]] = []

    def dfs(node: TreeNode | None, row: int, col: int) -> None:
        if node:
            cells.append((col, row, node.val))
            dfs(node.left, row + 1, col - 1)
            dfs(node.right, row + 1, col + 1)

    dfs(root, 0, 0)
    # tuple order = column, then row, then value
    out: dict[int, list[int]] = defaultdict(list)
    for col, _, val in sorted(cells):
        out[col].append(val)
    return list(out.values())


assert vertical_traversal(build_tree([1, 2, 3, 4, 6, 5, 7])) == [
    [4],
    [2],
    [1, 5, 6],
    [3],
    [7],
]
print(vertical_traversal(build_tree([3, 9, 20, None, None, 15, 7])))

[[9], [3, 15], [20], [7]]


---
# 3) Binary Search Tree

**In plain English.** A binary search tree keeps every smaller key on the left and every
larger key on the right, at every node. So you can find things by going one way at each
step, and reading it left-to-right (in-order) gives the keys in sorted order.

> **Analogy.** A phone book you can only open in the middle: each look tells you whether
> to go to the left or right half, and you never need both.

Two consequences carry almost every question:

| Fact | Use it for |
|---|---|
| **In-order traversal is sorted** | k-th smallest, validate, minimum difference, recover swapped nodes, iterators |
| **Each lookup goes one way** (O(h)) | search, insert, delete, LCA, successor, range pruning |

**Recognition signals**

- "binary search tree", "BST", "valid BST"
- "k-th smallest", "successor", "closest value", "sorted order"
- "insert", "delete", "range sum between low and high"
- "convert sorted array / list to a balanced tree"

**Common bugs**

- Validating only parent–child pairs: every node must respect *all* ancestors' bounds.
- Using non-strict comparisons when the problem says keys are strictly ordered.
- Recursion on a skewed BST (h = n) — iterative traversal avoids stack overflow.
- Deleting a node with two children by simply removing it (must replace with successor).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Binary tree DFS | no ordering, so you must search both sides |
| Heap | heap order is parent ≤ children only; no left/right ordering, no sorted traversal |
| Binary search on arrays | same "go one way" idea, but O(log n) is guaranteed; a BST is only O(h) |

**Learning path.** Validate (bounds) → k-th smallest (stopping in-order) → LCA by value →
adjacent differences → iterator (paused in-order) → successor by descent → build balanced →
insert → delete (three cases) → range pruning → recover (find the drops) → convert to a
doubly linked list.

## 3.1 BST properties and in-order traversal

Choose one of two tools: **descend** (compare and go one way, O(h)) or **in-order**
(visit in sorted order, often with a `prev` pointer to compare neighbours).

### 3.1.1 [98. Validate Binary Search Tree](https://leetcode.com/problems/validate-binary-search-tree/) — ★★★ · Medium

**Problem.** Is the tree a valid BST (strict inequalities)?  
**Signal.** Each node must lie in an open interval inherited from its ancestors.  
*New here: Passing a valid range (low, high) down the tree.*

**Fine print.** Ask before coding:
- Strict, or are duplicates allowed? → Strict, so duplicates are invalid.
- Values at the 32-bit limits? → Yes, so use ±∞ (or `None`) as the bounds.

> 💡 **Key insight.** Every ancestor constrains a node, so the node must lie inside the interval accumulated from the root, not just compare with its parent.

**Brute force →** For each node, check it against max of the left subtree and min of the right → O(n²). Bounds passed down → O(n).

**Steps**
1. Start with `(−∞, +∞)`.
2. A node outside `(low, high)` → invalid.
3. Left child gets `(low, node.val)`; right child gets `(node.val, high)`.
4. Empty subtrees are valid.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

⚠️ **Trap.** `[5, 4, 6, None, None, 3, 7]`: 3 is a fine child of 6 but violates the root bound 5.

In [42]:
def is_valid_bst(root: TreeNode | None) -> bool:
    """True if the tree satisfies the BST property."""

    def valid(node: TreeNode | None, low: float, high: float) -> bool:
        if not node:
            return True
        # must respect every ancestor, summarised by (low, high)
        if not low < node.val < high:
            return False
        # left side caps at node.val, right side floors at it
        return valid(node.left, low, node.val) and valid(node.right, node.val, high)

    return valid(root, float("-inf"), float("inf"))


assert is_valid_bst(build_tree([2, 1, 3]))
assert not is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6]))
assert not is_valid_bst(build_tree([5, 4, 6, None, None, 3, 7]))
print(is_valid_bst(build_tree([2, 1, 3])))

True


### 3.1.2 [230. Kth Smallest Element in a BST](https://leetcode.com/problems/kth-smallest-element-in-a-bst/) — ★★★ · Medium

**Problem.** k-th smallest key in a BST.  
**Signal.** In-order = sorted → stop at the k-th visit.  
*New here: Iterative in-order with an explicit stack, stopping early.*

**Fine print.** Ask before coding:
- 1-indexed `k` that is always valid? → Yes.
- Follow-up: frequent inserts and deletes? → Store subtree sizes in each node.

> 💡 **Key insight.** In-order visits keys in ascending order, so the k-th node visited is the answer.

**Brute force →** Collect all values in order, then index → O(n). Stopping early → O(h + k).

**Steps**
1. Push the left spine onto a stack.
2. Pop a node: it is the next smallest; decrement `k`.
3. If `k` hits 0, return it.
4. Otherwise move to its right child and push that left spine.

**Complexity**

- **Time:** O(h + k).
- **Space:** O(h).

In [43]:
def kth_smallest(root: TreeNode | None, k: int) -> int:
    """k-th smallest value in a BST."""
    stack: list[TreeNode] = []
    node = root
    while True:
        # go as far left as possible
        while node:
            stack.append(node)
            node = node.left
        # next node in sorted order
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        # continue with the larger keys
        node = node.right


assert kth_smallest(build_tree([3, 1, 4, None, 2]), 1) == 1
assert kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3
print(kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3))

3


### 3.1.3 [235. Lowest Common Ancestor of a Binary Search Tree](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-search-tree/) — ★★★ · Medium

**Problem.** LCA of `p` and `q` in a BST.  
**Signal.** The LCA is the first node whose value lies between `p` and `q`.  
*New here: Using ordering to turn LCA into a one-way descent.*

**Fine print.** Ask before coding:
- Are `p` and `q` guaranteed to exist? → Yes.
- Unique values? → Yes.

> 💡 **Key insight.** While both targets are on the same side of a node, the LCA is on that side too; the first node that separates them (or equals one) is the LCA.

**Brute force →** The general binary-tree LCA → O(n). Using BST order → O(h).

**Steps**
1. Both values smaller than the node → go left.
2. Both larger → go right.
3. Otherwise return the node.

**Complexity**

- **Time:** O(h).
- **Space:** O(1).

In [44]:
def lca_bst(root: TreeNode, p: TreeNode, q: TreeNode) -> TreeNode:
    """LCA in a BST."""
    node = root
    while True:
        # both targets lie left
        if p.val < node.val and q.val < node.val:
            node = node.left
        # both targets lie right
        elif p.val > node.val and q.val > node.val:
            node = node.right
        # they split here (or one of them is here)
        else:
            return node


t = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
assert lca_bst(t, find(t, 2), find(t, 8)).val == 6
assert lca_bst(t, find(t, 2), find(t, 4)).val == 2
print(lca_bst(t, find(t, 3), find(t, 5)).val)

4


### 3.1.4 [530. Minimum Absolute Difference in BST](https://leetcode.com/problems/minimum-absolute-difference-in-bst/) — ★★ · Easy

**Problem.** Minimum difference between values of any two nodes.  
**Signal.** In sorted order, the closest pair is adjacent → in-order with a `prev` value.  
*New here: In-order with a `prev` value to compare neighbours.*

**Fine print.** Ask before coding:
- At least two nodes? → Yes.
- Unique values? → Yes, so the answer is ≥ 1.

> 💡 **Key insight.** In a sorted sequence the smallest gap is between two neighbours, so compare each in-order value with the previous one.

**Brute force →** Compare all pairs → O(n²). In-order neighbours → O(n).

**Steps**
1. In-order traversal.
2. At each node, compare with `prev` and update the best gap.
3. Set `prev` to the current value.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [45]:
def get_minimum_difference(root: TreeNode | None) -> int:
    """Smallest gap between two BST values."""
    best, prev = float("inf"), None

    def inorder(node: TreeNode | None) -> None:
        nonlocal best, prev
        if not node:
            return
        inorder(node.left)
        # neighbours in sorted order give the candidate gaps
        if prev is not None:
            best = min(best, node.val - prev)
        prev = node.val
        inorder(node.right)

    inorder(root)
    return best


assert get_minimum_difference(build_tree([4, 2, 6, 1, 3])) == 1
assert get_minimum_difference(build_tree([1, 0, 48, None, None, 12, 49])) == 1
print(get_minimum_difference(build_tree([4, 2, 6, 1, 3])))

1


### 3.1.5 [173. Binary Search Tree Iterator](https://leetcode.com/problems/binary-search-tree-iterator/) — ★★ · Medium

**Problem.** Iterator over a BST in ascending order with O(1) amortised `next` and O(h) memory.  
**Signal.** Iterative in-order traversal, paused between calls.  
*New here: Splitting an iterative traversal across method calls.*

**Fine print.** Ask before coding:
- Will `next` only be called when `has_next` is true? → Yes.
- O(h) memory? → Yes, which rules out flattening the whole tree first.

> 💡 **Key insight.** The stack of the iterative in-order traversal is the iterator's whole state; each `next` is one step of that loop.

**Brute force →** Precompute the full in-order list → O(n) memory. The lazy stack keeps O(h).

**Steps**
1. Constructor pushes the left spine of the root.
2. `next` pops a node (the next smallest).
3. Push the left spine of its right child.
4. `has_next` checks whether the stack is non-empty.

**Complexity**

- **Time:** O(1) amortised per call — each node is pushed and popped once.
- **Space:** O(h).

In [46]:
class BSTIterator:
    """Lazy in-order iterator."""

    def __init__(self, root: TreeNode | None) -> None:
        self.stack: list[TreeNode] = []
        self._push_left(root)

    def _push_left(self, node: TreeNode | None) -> None:
        # smallest unvisited keys sit along the left spine
        while node:
            self.stack.append(node)
            node = node.left

    def next(self) -> int:
        node = self.stack.pop()
        # the right subtree holds the next larger keys
        self._push_left(node.right)
        return node.val

    def has_next(self) -> bool:
        return bool(self.stack)


it = BSTIterator(build_tree([7, 3, 15, None, None, 9, 20]))
out = []
while it.has_next():
    out.append(it.next())
assert out == [3, 7, 9, 15, 20]
print(out)

[3, 7, 9, 15, 20]


### 3.1.6 [285. Inorder Successor in BST](https://leetcode.com/problems/inorder-successor-in-bst/) — ★★ · Medium · Premium

**Problem.** The node with the smallest key greater than `p.val`.  
**Signal.** BST descent remembering the last node where we went left.  
*New here: Descent that remembers a best candidate so far.*

**Fine print.** Ask before coding:
- No successor? → Return `None`.
- Unique values? → Yes.

> 💡 **Key insight.** Every node larger than `p` is a candidate, and going left from it can only find a smaller (better) candidate.

**Brute force →** Full in-order traversal and take the node after `p` → O(n). Descent → O(h).

**Steps**
1. Start at the root with no candidate.
2. `p.val < node.val` → record the node, go left.
3. Otherwise go right.
4. Return the last recorded candidate.

**Complexity**

- **Time:** O(h).
- **Space:** O(1).

In [47]:
def inorder_successor(root: TreeNode | None, p: TreeNode) -> TreeNode | None:
    """Next larger node in a BST."""
    succ, node = None, root
    while node:
        # larger than p: a candidate; a smaller one may be to the left
        if p.val < node.val:
            succ, node = node, node.left
        # not larger: the successor is to the right
        else:
            node = node.right
    return succ


t = build_tree([5, 3, 6, 2, 4, None, None, 1])
assert inorder_successor(t, find(t, 4)).val == 5
assert inorder_successor(t, find(t, 6)) is None
print(inorder_successor(t, find(t, 1)).val)

2


### 3.1.7 [108. Convert Sorted Array to Binary Search Tree](https://leetcode.com/problems/convert-sorted-array-to-binary-search-tree/) — ★★ · Easy

**Problem.** Build a height-balanced BST from a sorted array.  
**Signal.** The middle element is the root; recurse on halves.  
*New here: Building a BST by divide and conquer.*

**Fine print.** Ask before coding:
- Any balanced BST is fine? → Yes, several answers are valid.
- Strictly increasing input? → Yes.

> 💡 **Key insight.** Picking the middle as the root splits the rest into two halves whose sizes differ by at most one, which keeps every level balanced.

**Brute force →** Insert values one by one → a skewed O(n)-height tree on sorted input. Middle-as-root → height O(log n).

**Steps**
1. Empty range → `None`.
2. Root = middle element.
3. Left subtree from the left half, right subtree from the right half.

**Complexity**

- **Time:** O(n).
- **Space:** O(log n) recursion.

In [48]:
def sorted_array_to_bst(nums: list[int]) -> TreeNode | None:
    """Balanced BST from sorted values."""

    def make(lo: int, hi: int) -> TreeNode | None:
        if lo > hi:
            return None
        # middle value as root keeps both sides the same size
        mid = (lo + hi) // 2
        return TreeNode(nums[mid], make(lo, mid - 1), make(mid + 1, hi))

    return make(0, len(nums) - 1)


t = sorted_array_to_bst([-10, -3, 0, 5, 9])
assert is_valid_bst(t) and is_balanced(t)
print(tree_to_list(t))

[0, -10, 5, None, -3, None, 9]


### 3.1.8 [701. Insert into a Binary Search Tree](https://leetcode.com/problems/insert-into-a-binary-search-tree/) — ★ · Medium

**Problem.** Insert a value into a BST (value not present); return the root.  
**Signal.** Walk down like a search; attach at the empty slot.  
*New here: Modifying the tree at the point where a search fails.*

**Fine print.** Ask before coding:
- Is the value guaranteed not to be present? → Yes.
- Any valid BST is fine? → Yes.

> 💡 **Key insight.** A failed search ends at exactly the empty slot where the value belongs.

**Brute force →** Rebuild the tree from all values → O(n). Search to the slot → O(h).

**Steps**
1. Empty tree → new node.
2. Compare and choose left or right.
3. If that child is empty, place the new node there.
4. Otherwise continue from the child.

**Complexity**

- **Time:** O(h).
- **Space:** O(1).

In [49]:
def insert_into_bst(root: TreeNode | None, val: int) -> TreeNode:
    """BST with val inserted."""
    if not root:
        return TreeNode(val)
    node = root
    while True:
        side = "left" if val < node.val else "right"
        child = getattr(node, side)
        # the search fell off here: this is the slot
        if not child:
            setattr(node, side, TreeNode(val))
            return root
        node = child


t = insert_into_bst(build_tree([4, 2, 7, 1, 3]), 5)
assert tree_to_list(t) == [4, 2, 7, 1, 3, 5]
print(tree_to_list(t))

[4, 2, 7, 1, 3, 5]


### 3.1.9 [450. Delete Node in a BST](https://leetcode.com/problems/delete-node-in-a-bst/) — ★★ · Medium

**Problem.** Delete the node with key `key` and return the root.  
**Signal.** Three cases: leaf, one child, two children (replace with in-order successor).  
*New here: The three-case deletion, using the in-order successor.*

**Fine print.** Ask before coding:
- Key not found? → Return the tree unchanged.
- Replace with the successor or the predecessor? → Either.
- Must the height stay O(h)? → Yes.

> 💡 **Key insight.** A node with two children can take its successor's value (the smallest in its right subtree), and that successor is easy to delete because it has no left child.

**Brute force →** Rebuild from the remaining values → O(n). The three-case delete → O(h).

**Steps**
1. Recurse left or right to find the key.
2. Zero or one child → return that child (or `None`).
3. Two children → find the smallest node in the right subtree.
4. Copy its value up and delete it from the right subtree.

**Complexity**

- **Time:** O(h).
- **Space:** O(h) recursion.

In [50]:
def delete_node(root: TreeNode | None, key: int) -> TreeNode | None:
    """BST with key removed."""
    if not root:
        return None
    if key < root.val:
        root.left = delete_node(root.left, key)
    elif key > root.val:
        root.right = delete_node(root.right, key)
    else:
        # at most one child: splice it in
        if not root.left or not root.right:
            return root.left or root.right
        # two children: borrow the in-order successor's value
        succ = root.right
        while succ.left:
            succ = succ.left
        root.val = succ.val
        # the successor has no left child, so this delete is easy
        root.right = delete_node(root.right, succ.val)
    return root


t = delete_node(build_tree([5, 3, 6, 2, 4, None, 7]), 3)
assert tree_to_list(t) == [5, 4, 6, 2, None, None, 7]
assert tree_to_list(delete_node(build_tree([5, 3, 6, 2, 4, None, 7]), 0)) == [
    5,
    3,
    6,
    2,
    4,
    None,
    7,
]
print(tree_to_list(t))

[5, 4, 6, 2, None, None, 7]


### 3.1.10 [938. Range Sum of BST](https://leetcode.com/problems/range-sum-of-bst/) — ★ · Easy

**Problem.** Sum of values in `[low, high]`.  
**Signal.** Prune subtrees the BST property rules out.  
*New here: Pruning: skipping a whole subtree that cannot contribute.*

**Fine print.** Ask before coding:
- Are `low` and `high` inclusive? → Yes.
- Unique values? → Yes.

> 💡 **Key insight.** If a node is below `low`, its entire left subtree is too; if above `high`, its entire right subtree is too.

**Brute force →** Visit every node and test the range → O(n) always. Pruning visits only the relevant part.

**Steps**
1. Below `low` → only search the right subtree.
2. Above `high` → only search the left subtree.
3. In range → add it and search both sides.

**Complexity**

- **Time:** O(n) worst case, much less when the range is narrow.
- **Space:** O(h).

In [51]:
def range_sum_bst(root: TreeNode | None, low: int, high: int) -> int:
    """Sum of BST values within [low, high]."""
    if not root:
        return 0
    # too small: the whole left subtree is too small as well
    if root.val < low:
        return range_sum_bst(root.right, low, high)
    # too large: the whole right subtree is too large as well
    if root.val > high:
        return range_sum_bst(root.left, low, high)
    return (
        root.val
        + range_sum_bst(root.left, low, high)
        + range_sum_bst(root.right, low, high)
    )


assert range_sum_bst(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15) == 32
assert range_sum_bst(build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6]), 6, 10) == 23
print(range_sum_bst(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15))

32


### 3.1.11 [99. Recover Binary Search Tree](https://leetcode.com/problems/recover-binary-search-tree/) — ★ · Medium

**Problem.** Exactly two nodes were swapped; fix the BST without changing its structure.  
**Signal.** In-order sequence has one or two "drops" where `prev > cur`.  
*New here: Detecting a swap from drops in a sorted sequence.*

**Fine print.** Ask before coding:
- Exactly two nodes swapped? → Yes.
- O(1) space? → The follow-up asks for Morris traversal.

> 💡 **Key insight.** Swapping two values in a sorted sequence creates a drop where the large one sits and a drop where the small one sits (one drop if they were adjacent).

**Brute force →** Collect in-order values, sort them, and write them back → O(n log n) time, O(n) space. Finding the drops → O(n) time, O(h) space.

**Steps**
1. In-order traversal with `prev`.
2. At the first drop, `first = prev`.
3. At every drop, `second = cur` (the last one wins).
4. Swap the values of `first` and `second`.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [52]:
def recover_tree(root: TreeNode | None) -> None:
    """Swap back the two misplaced values in place."""
    first = second = prev = None

    def inorder(node: TreeNode | None) -> None:
        nonlocal first, second, prev
        if not node:
            return
        inorder(node.left)
        # a drop in what should be ascending order
        if prev and prev.val > node.val:
            # the first drop exposes the too-large value
            if not first:
                first = prev
            # the last drop exposes the too-small value
            second = node
        prev = node
        inorder(node.right)

    inorder(root)
    first.val, second.val = second.val, first.val


t = build_tree([1, 3, None, None, 2])
recover_tree(t)
assert tree_to_list(t) == [3, 1, None, None, 2]
t = build_tree([3, 1, 4, None, None, 2])
recover_tree(t)
assert tree_to_list(t) == [2, 1, 4, None, None, 3]
print(tree_to_list(t))

[2, 1, 4, None, None, 3]


### 3.1.12 [426. Convert Binary Search Tree to Sorted Doubly Linked List](https://leetcode.com/problems/convert-binary-search-tree-to-sorted-doubly-linked-list/) — ★ · Medium · Premium

**Problem.** Convert a BST in place into a sorted *circular* doubly linked list (`left` = prev, `right` = next); return the smallest node.  
**Signal.** In-order traversal while linking each node to the previously visited one.  
*New here: Rewiring tree pointers during an in-order walk.*

**Fine print.** Ask before coding:
- Circular, so head and tail link to each other? → Yes.
- In place? → Yes, reuse `left`/`right` as prev/next.
- Empty tree? → Return `None`.

> 💡 **Key insight.** In-order visits nodes in list order, so linking each node to the previously visited one builds the list as you go.

**Brute force →** Collect in-order nodes into an array and link neighbours → O(n) extra space. Linking during traversal needs only O(h).

**Steps**
1. In-order traversal with `first` and `last`.
2. On a node: if `last` exists, link `last.right = node`, `node.left = last`; else it is `first`.
3. Set `last = node`.
4. After the traversal, link `last` and `first` to close the circle.

**Complexity**

- **Time:** O(n).
- **Space:** O(h).

In [53]:
def tree_to_doubly_list(root: TreeNode | None) -> TreeNode | None:
    """Head of the circular sorted doubly linked list."""
    if not root:
        return None
    first = last = None

    def inorder(node: TreeNode | None) -> None:
        nonlocal first, last
        if not node:
            return
        inorder(node.left)
        # link the previous node in sorted order to this one
        if last:
            last.right, node.left = node, last
        else:
            first = node
        last = node
        inorder(node.right)

    inorder(root)
    # close the circle
    last.right, first.left = first, last
    return first


head = tree_to_doubly_list(build_tree([4, 2, 5, 1, 3]))
vals, node = [], head
for _ in range(5):
    vals.append(node.val)
    node = node.right
assert vals == [1, 2, 3, 4, 5] and node is head and head.left.val == 5
print(vals)

[1, 2, 3, 4, 5]


---
# 4) Trie

**In plain English.** A trie stores words letter by letter along paths from a root. Words
that share a beginning share the same path, so every node *is* a prefix.

> **Analogy.** A dictionary's thumb tabs: open at "c", then "ca", then "cat" — every
> step narrows to the words with that beginning, no matter how many words the book holds.

Lookups cost O(length of the word), independent of how many words are stored. Here each
node is a `dict` of children plus an end marker `"$"`.

**Recognition signals**

- "prefix", "starts with", "autocomplete", "search suggestions"
- many words searched at once (Word Search II) — share work across common prefixes
- wildcard search (`.` matches any letter)
- "replace each word by its shortest root"
- maximum XOR — a trie over *bits*

**Common bugs**

- Forgetting the end marker: `"app"` is a prefix of `"apple"` but not a stored word.
- Iterating the `"$"` key as if it were a child during wildcard search.
- Reporting the same word twice in a grid search (remove the marker once found).
- Not restoring a grid cell after DFS (`board[r][c] = ch`).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Hash set of words | O(L) exact lookup too, but no shared prefixes, no "starts with" |
| Sorted list + binary search | also answers prefix queries (Search Suggestions) with less code |
| Backtracking on a grid | Word Search II = grid backtracking *guided* by a trie |

**Learning path.** Basic trie → wildcard DFS → trie-guided grid search → prefix
suggestions (sorted-list alternative) → shortest-root replacement → bitwise trie for XOR.

## 4.1 Prefix trees

Each question either builds the trie once and walks it per query (Replace Words, Max XOR)
or walks the trie *alongside* another search (wildcards, the grid).

### 4.1.1 [208. Implement Trie (Prefix Tree)](https://leetcode.com/problems/implement-trie-prefix-tree/) — ★★★ · Medium

**Problem.** `insert`, `search` (whole word) and `starts_with` (prefix).  
**Signal.** Nested dicts, one level per character, with an end-of-word marker.  
*New here: The trie itself: nested dicts and an end marker.*

**Fine print.** Ask before coding:
- Only lowercase letters? → Yes, so a 26-child array or a dict both work.
- Does `search` need the word to have been inserted, not just a prefix? → Yes, the end-of-word flag.

> 💡 **Key insight.** A word is a path from the root; "is a stored word" needs an end marker at the path's last node, "is a prefix" only needs the path to exist.

**Brute force →** A list of words with `startswith` checks → O(n · L) per query. The trie → O(L).

**Steps**
1. `insert`: walk/create one child per character, then set `"$"`.
2. `_walk(s)`: follow characters, returning `None` if a child is missing.
3. `search`: the walk ends at a node with `"$"`.
4. `starts_with`: the walk merely succeeds.

**Complexity**

- **Time:** O(L) per operation.
- **Space:** O(total characters inserted).

In [54]:
class Trie:
    """Prefix tree over lowercase words."""

    def __init__(self) -> None:
        self.root: dict = {}

    def insert(self, word: str) -> None:
        node = self.root
        # create the path one character at a time
        for ch in word:
            node = node.setdefault(ch, {})
        # mark that a whole word ends here
        node["$"] = True

    def _walk(self, s: str) -> dict | None:
        node = self.root
        for ch in s:
            if ch not in node:
                return None
            node = node[ch]
        return node

    def search(self, word: str) -> bool:
        # path exists and a word ends exactly here
        node = self._walk(word)
        return bool(node and node.get("$"))

    def starts_with(self, prefix: str) -> bool:
        # any path at all is enough
        return self._walk(prefix) is not None


trie = Trie()
trie.insert("apple")
assert trie.search("apple") and not trie.search("app") and trie.starts_with("app")
trie.insert("app")
assert trie.search("app")
print("ok")

ok


### 4.1.2 [211. Design Add and Search Words Data Structure](https://leetcode.com/problems/design-add-and-search-words-data-structure/) — ★★★ · Medium

**Problem.** `add_word(word)`; `search(pattern)` where `.` matches any letter.  
**Signal.** Trie + DFS that branches on `.`.  
*New here: Branching through a trie (DFS over children).*

**Fine print.** Ask before coding:
- Can `.` match any letter but not an empty one? → Yes.
- How many dots per query? → At most 2 on LeetCode, so branching stays small.

> 💡 **Key insight.** A normal letter follows one edge; a `.` must try every edge, which is a small DFS over the trie.

**Brute force →** Compare the pattern with every stored word → O(n · L). Trie DFS only explores matching prefixes.

**Steps**
1. `add_word` is a normal trie insert.
2. `dfs(node, i)`: at the end of the pattern, check `"$"`.
3. `.` → try every child (skipping the `"$"` key).
4. A letter → follow that child if it exists.

**Complexity**

- **Time:** O(L) to add; search is O(26^dots · L) in the worst case.
- **Space:** O(total characters) + O(L) recursion.

In [55]:
class WordDictionary:
    """Trie supporting '.' wildcards."""

    def __init__(self) -> None:
        self.root: dict = {}

    def add_word(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.setdefault(ch, {})
        node["$"] = True

    def search(self, word: str) -> bool:
        def dfs(node: dict, i: int) -> bool:
            # consumed the pattern: a word must end here
            if i == len(word):
                return "$" in node
            # wildcard: any child may continue the match
            if word[i] == ".":
                return any(
                    dfs(child, i + 1) for key, child in node.items() if key != "$"
                )
            # literal: exactly one edge to follow
            return word[i] in node and dfs(node[word[i]], i + 1)

        return dfs(self.root, 0)


wd = WordDictionary()
for w in ["bad", "dad", "mad"]:
    wd.add_word(w)
assert (
    not wd.search("pad") and wd.search("bad") and wd.search(".ad") and wd.search("b..")
)
print("ok")

ok


### 4.1.3 [212. Word Search II](https://leetcode.com/problems/word-search-ii/) — ★★★ · Hard

**Problem.** All dictionary words that can be traced in a letter grid (adjacent cells, no reuse).  
**Signal.** Many words over one grid → trie of words + one DFS per cell.  
*New here: A trie guiding a grid backtracking search, with pruning.*

**Fine print.** Ask before coding:
- Report each word once even if found several times? → Yes.
- Can a cell be reused within one word? → No.
- Unique dictionary words? → Yes.

> 💡 **Key insight.** Walking the grid and the trie together searches for *all* words at once, and a missing trie edge kills a dead prefix immediately.

**Brute force →** Run single-word Word Search for each word → O(words · cells · 3^L). One trie-guided DFS shares work across common prefixes.

**Steps**
1. Insert all words into a trie, storing the whole word at its end node.
2. From each cell whose letter starts a trie edge, DFS: move into the child node.
3. If the node holds a word, record it and remove it (no duplicates).
4. Mark the cell visited, explore neighbours that continue the trie, restore it; prune empty trie nodes.

**Complexity**

- **Time:** O(cells · 4 · 3^(L−1)) worst case; far less with pruning.
- **Space:** O(total characters) for the trie + O(L) recursion.

In [56]:
def find_words(board: list[list[str]], words: list[str]) -> list[str]:
    """Dictionary words present in the grid."""
    # trie with the full word stored at its end node
    root: dict = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    rows, cols = len(board), len(board[0])
    found: list[str] = []

    def dfs(r: int, c: int, parent: dict) -> None:
        ch = board[r][c]
        node = parent[ch]
        # a complete word: record it once
        if "$" in node:
            found.append(node.pop("$"))
        # mark visited, explore neighbours that extend a trie path
        board[r][c] = "#"
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < rows and 0 <= nc < cols and board[nr][nc] in node:
                dfs(nr, nc, node)
        board[r][c] = ch
        # prune trie branches with nothing left to find
        if not node:
            parent.pop(ch)

    for r in range(rows):
        for c in range(cols):
            if board[r][c] in root:
                dfs(r, c, root)
    return found


board = [list("oaan"), list("etae"), list("ihkr"), list("iflv")]
assert sorted(find_words(board, ["oath", "pea", "eat", "rain"])) == ["eat", "oath"]
assert find_words([list("ab"), list("cd")], ["abcb"]) == []
print(find_words(board, ["oath", "pea", "eat", "rain"]))

['oath', 'eat']


### 4.1.4 [1268. Search Suggestions System](https://leetcode.com/problems/search-suggestions-system/) — ★ · Medium

**Problem.** After each typed character, return up to three lexicographically smallest products with that prefix.  
**Signal.** Sorted list + binary search on the prefix (or a trie storing top-3 per node).  
*New here: Prefix search *without* a trie — a sorted list does the same job.*

**Fine print.** Ask before coding:
- Return results after each character typed? → Yes.
- At most three, in lexicographic order? → Yes.

> 💡 **Key insight.** In a sorted list, all words with a given prefix form one contiguous block starting at `bisect_left(prefix)`.

**Brute force →** Filter all products for every prefix → O(L · n · L). Sort once + binary search → O(n log n + L log n).

**Steps**
1. Sort the products.
2. For each typed prefix, find the first position ≥ prefix with `bisect_left`.
3. Take up to three products from there that still start with the prefix.

**Complexity**

- **Time:** O(n log n + L log n) (plus prefix checks).
- **Space:** O(n) for the sorted copy.

In [57]:
def suggested_products(products: list[str], search_word: str) -> list[list[str]]:
    """Top-3 suggestions per typed prefix."""
    products = sorted(products)
    out, prefix = [], ""
    for ch in search_word:
        prefix += ch
        # words with this prefix start here, in sorted order
        i = bisect_left(products, prefix)
        out.append([p for p in products[i : i + 3] if p.startswith(prefix)])
    return out


res = suggested_products(
    ["mobile", "mouse", "moneypot", "monitor", "mousepad"], "mouse"
)
assert res[0] == ["mobile", "moneypot", "monitor"]
assert res[-1] == ["mouse", "mousepad"]
print(res)

[['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad'], ['mouse', 'mousepad'], ['mouse', 'mousepad']]


### 4.1.5 [648. Replace Words](https://leetcode.com/problems/replace-words/) — ★ · Medium

**Problem.** Replace every word in a sentence by its shortest dictionary root that is a prefix of it.  
**Signal.** Prefix lookup per word → trie, stop at the first end marker.  
*New here: Stopping at the *first* end marker on a walk.*

**Fine print.** Ask before coding:
- Several roots match? → Use the shortest.
- Words separated by single spaces? → Yes.

> 💡 **Key insight.** Walking a word down the root trie, the first end marker you meet is its shortest root.

**Brute force →** For each word, test every root with `startswith` → O(words · roots · L). The trie → O(total characters).

**Steps**
1. Insert every root into a trie.
2. For each word, walk the trie letter by letter.
3. Hit an end marker → return the prefix so far.
4. Missing edge or no marker → keep the word.

**Complexity**

- **Time:** O(total characters).
- **Space:** O(total root characters).

In [58]:
def replace_words(dictionary: list[str], sentence: str) -> str:
    """Sentence with words shortened to their roots."""
    root: dict = {}
    for w in dictionary:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True

    def shorten(word: str) -> str:
        node = root
        for i, ch in enumerate(word):
            # no root continues this way
            if ch not in node:
                return word
            node = node[ch]
            # first root found is the shortest
            if "$" in node:
                return word[: i + 1]
        return word

    return " ".join(shorten(w) for w in sentence.split())


out = replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery")
assert out == "the cat was rat by the bat"
print(out)

the cat was rat by the bat


### 4.1.6 [421. Maximum XOR of Two Numbers in an Array](https://leetcode.com/problems/maximum-xor-of-two-numbers-in-an-array/) — ★ · Medium

**Problem.** Maximum `a XOR b` over pairs.  
**Signal.** Bitwise trie: greedily choose the opposite bit from the top.  
*New here: A trie over bits, walked greedily.*

**Fine print.** Ask before coding:
- Non-negative values? → Yes, up to 2³¹ − 1, so 31 bits.
- Can `a` and `b` be the same element? → Yes, giving 0.

> 💡 **Key insight.** A 1 in a higher bit beats any combination of lower bits, so for each number, walk from the top bit and always take the opposite bit if some stored number has it.

**Brute force →** All pairs → O(n²). Bitwise trie → O(n · 31).

**Steps**
1. Insert every number as 31 bits, most significant first.
2. For each number, walk the trie from the top bit.
3. Prefer the child with the opposite bit (that XOR bit becomes 1).
4. Track the best XOR found.

**Complexity**

- **Time:** O(n · 31).
- **Space:** O(n · 31) trie nodes.

In [59]:
def find_maximum_xor(nums: list[int]) -> int:
    """Largest XOR of any pair."""
    # binary trie, most significant bit first
    root: dict = {}
    for x in nums:
        node = root
        for b in range(30, -1, -1):
            node = node.setdefault((x >> b) & 1, {})
    best = 0
    for x in nums:
        node, cur = root, 0
        for b in range(30, -1, -1):
            bit = (x >> b) & 1
            # opposite bit available: this XOR bit becomes 1
            if 1 - bit in node:
                cur |= 1 << b
                node = node[1 - bit]
            else:
                node = node[bit]
        best = max(best, cur)
    return best


assert find_maximum_xor([3, 10, 5, 25, 2, 8]) == 28
assert find_maximum_xor([14, 70, 53, 83, 49, 91, 36, 80, 92, 51, 66, 70]) == 127
print(find_maximum_xor([3, 10, 5, 25, 2, 8]))

28


---
# 5) Heap / Priority Queue

**In plain English.** A heap always hands you the smallest item in O(log n) per insert or
removal, without keeping everything fully sorted.

> **Analogy.** A hospital triage queue: patients arrive in any order, but the most urgent
> one is always seen next. Nobody sorts the whole waiting room — they just keep track of
> who is most urgent.

Python's `heapq` is a **min-heap** on a list; push `-x` for a max-heap; tuples compare
element by element, so `(priority, tie_breaker, item)` works.

| Goal | Heap | Size |
|---|---|---|
| k largest | **min**-heap, pop when size > k (the top is the k-th largest) | k |
| k smallest | **max**-heap (push negatives) | k |
| repeatedly take the best | min/max heap of candidates | all candidates |
| median of a stream | max-heap for the lower half, min-heap for the upper half | n |
| merge k sorted sources | min-heap of the current head of each source | k |

**Recognition signals**

- "k largest / smallest / closest / most frequent", "k-th largest"
- "repeatedly take the two largest", "always process the cheapest / shortest next"
- "stream", "running median"
- "merge k sorted …", "k pairs with smallest sums", "smallest range covering k lists"
- greedy with changing availability (IPO, CPU scheduling, ladders)

**Common bugs**

- Using a max-heap for "k largest" (it should be a min-heap of size k).
- Forgetting to negate on the way *out* of a max-heap.
- Pushing objects that are not comparable on ties — add an index tie-breaker.
- `heapq.heappush` on an un-heapified list (call `heapify` first, O(n)).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Sorting | O(n log n) once; a heap supports inserts and removals as you go |
| Quickselect | k-th element in O(n) average, but not streaming |
| Bucket sort (Top K Frequent) | O(n) when keys are bounded small integers |
| Monotonic deque | max/min of a *sliding window* — elements leave in order of arrival |

**Learning path.** Size-k heap (k-th largest in a stream) → max-heap simulation (stones) →
k closest → quickselect alternative → composite keys (top k words) → k-way merge (k pairs,
ugly numbers). Then scheduling: counting formula (task scheduler) → greedy placement
(reorganize) → feed merge (Twitter) → two heaps (median) → greedy with availability (IPO,
CPU) → regret (ladders) → k-list window (smallest range).

## 5.1 Top-k and k-th element

For "top k", keep a heap of size k whose top is the *worst* item you are keeping — then
one comparison decides whether a new item gets in.

### 5.1.1 [703. Kth Largest Element in a Stream](https://leetcode.com/problems/kth-largest-element-in-a-stream/) — ★★ · Easy

**Problem.** After each `add`, return the k-th largest element so far.  
**Signal.** k-th largest → min-heap of size k.  
*New here: The size-k min-heap.*

**Fine print.** Ask before coding:
- Can the initial array have fewer than `k` elements? → Yes, but at least `k` exist whenever you query.
- Duplicates count separately? → Yes.

> 💡 **Key insight.** If you keep only the k largest values, the smallest of them is the k-th largest — the top of a min-heap.

**Brute force →** Sort on every add → O(n log n) per call. Size-k heap → O(log k).

**Steps**
1. Push the new value.
2. If the heap has more than k items, pop the smallest.
3. Return the top.

**Complexity**

- **Time:** O(log k) per add.
- **Space:** O(k).

In [60]:
class KthLargest:
    """Running k-th largest value."""

    def __init__(self, k: int, nums: list[int]) -> None:
        self.k = k
        # min-heap of the k largest values seen
        self.heap: list[int] = []
        for x in nums:
            self.add(x)

    def add(self, val: int) -> int:
        heapq.heappush(self.heap, val)
        # drop the smallest so only the top k remain
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        # smallest of the top k = k-th largest
        return self.heap[0]


kl = KthLargest(3, [4, 5, 8, 2])
assert [kl.add(v) for v in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]
print("ok")

ok


### 5.1.2 [1046. Last Stone Weight](https://leetcode.com/problems/last-stone-weight/) — ★★ · Easy

**Problem.** Repeatedly smash the two heaviest stones; return the last weight (or 0).  
**Signal.** Repeatedly take the two largest → max-heap.  
*New here: A max-heap via negation, used as a simulation.*

**Fine print.** Ask before coding:
- If two stones are equal? → Both are destroyed.
- No stones left? → Return 0.

> 💡 **Key insight.** The simulation only ever needs the two largest stones, which a max-heap provides in O(log n).

**Brute force →** Sort after every smash → O(n² log n). Heap → O(n log n).

**Steps**
1. Heapify the negated weights.
2. Pop the two heaviest.
3. If they differ, push back the difference.
4. Return the last stone or 0.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

In [61]:
def last_stone_weight(stones: list[int]) -> int:
    """Weight of the final stone, or 0."""
    # negate for a max-heap
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        # two heaviest stones
        a, b = -heapq.heappop(heap), -heapq.heappop(heap)
        if a != b:
            heapq.heappush(heap, -(a - b))
    return -heap[0] if heap else 0


assert last_stone_weight([2, 7, 4, 1, 8, 1]) == 1
assert last_stone_weight([1]) == 1
print(last_stone_weight([2, 7, 4, 1, 8, 1]))

1


### 5.1.3 [973. K Closest Points to Origin](https://leetcode.com/problems/k-closest-points-to-origin/) — ★★ · Medium

**Problem.** The `k` points closest to the origin.  
**Signal.** k smallest by distance → max-heap of size k (or `heapq.nsmallest`).  
*New here: "k smallest" uses the mirror heap: a max-heap of size k.*

**Fine print.** Ask before coding:
- Any output order? → Yes.
- Is the answer unique? → Yes, no ties at the boundary.
- Integer coordinates? → Yes, so compare squared distances and skip `sqrt`.

> 💡 **Key insight.** Keep the k closest points seen so far; the farthest of them sits at the top of a max-heap and is the one to drop.

**Brute force →** Sort by distance → O(n log n). Size-k max-heap → O(n log k). Quickselect → O(n) average.

**Steps**
1. Push `(−distance², point)`.
2. If the heap exceeds k, pop (removes the farthest).
3. Return the points left.

**Complexity**

- **Time:** O(n log k).
- **Space:** O(k).

In [62]:
def k_closest(points: list[list[int]], k: int) -> list[list[int]]:
    """k points nearest the origin."""
    # max-heap on distance (negated); squared distance avoids sqrt
    heap: list[tuple[int, list[int]]] = []
    for x, y in points:
        heapq.heappush(heap, (-(x * x + y * y), [x, y]))
        # drop the farthest of the kept points
        if len(heap) > k:
            heapq.heappop(heap)
    return [p for _, p in heap]


assert k_closest([[1, 3], [-2, 2]], 1) == [[-2, 2]]
assert sorted(k_closest([[3, 3], [5, -1], [-2, 4]], 2)) == [[-2, 4], [3, 3]]
print(k_closest([[3, 3], [5, -1], [-2, 4]], 2))

[[-2, 4], [3, 3]]


### 5.1.4 [215. Kth Largest Element in an Array](https://leetcode.com/problems/kth-largest-element-in-an-array/) — ★★ · Medium

**Problem.** k-th largest element (not distinct), ideally better than sorting.  
**Signal.** Min-heap of size k (O(n log k)) or Quickselect (O(n) average).  
*New here: Quickselect: partition and recurse into one side only.*

**Fine print.** Ask before coding:
- k-th largest in sorted order, not k-th distinct? → Sorted order.
- Can I avoid sorting? → Yes: a heap gives O(n log k), quickselect O(n) on average.

> 💡 **Key insight.** After partitioning around a pivot, you know exactly which part contains the k-th largest, so only that part needs further work.

**Brute force →** Sort → O(n log n). Size-k heap → O(n log k). Quickselect → O(n) average.

**Steps**
1. Pick a random pivot.
2. Split into greater / equal / less.
3. If k falls in "greater", recurse there; if in "equal", return the pivot.
4. Otherwise recurse into "less" with k reduced.

**Complexity**

- **Time:** O(n) average (each step keeps a fraction of the list), O(n²) worst.
- **Space:** O(n) for the lists here (in-place partitioning gets O(1)).

In [63]:
def find_kth_largest(nums: list[int], k: int) -> int:
    """k-th largest value via quickselect."""
    pivot = random.choice(nums)
    # three-way split handles duplicates
    greater = [x for x in nums if x > pivot]
    equal = [x for x in nums if x == pivot]
    # answer among the larger values
    if k <= len(greater):
        return find_kth_largest(greater, k)
    # answer is the pivot itself
    if k <= len(greater) + len(equal):
        return pivot
    # answer among the smaller values, with rank adjusted
    less = [x for x in nums if x < pivot]
    return find_kth_largest(less, k - len(greater) - len(equal))


assert find_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4
print(find_kth_largest([3, 2, 1, 5, 6, 4], 2))

5


### 5.1.5 [692. Top K Frequent Words](https://leetcode.com/problems/top-k-frequent-words/) — ★★ · Medium

**Problem.** `k` most frequent words; ties broken alphabetically.  
**Signal.** Top k with a composite key `(−count, word)`.  
*New here: A composite ordering key.*

**Fine print.** Ask before coding:
- How are ties broken? → Alphabetically.
- O(n log k)? → That is the follow-up.

> 💡 **Key insight.** The key `(−count, word)` orders by higher count first and alphabetically on ties, so the k smallest keys are the answer.

**Brute force →** Sort all words by the key → O(n log n). `nsmallest` with the key → O(n log k).

**Steps**
1. Count words.
2. `heapq.nsmallest(k, words, key=(−count, word))`.

**Complexity**

- **Time:** O(n log k).
- **Space:** O(n) for the counts.

In [64]:
def top_k_frequent_words(words: list[str], k: int) -> list[str]:
    """k most frequent words, alphabetical on ties."""
    count = Counter(words)
    # higher count first, then alphabetical
    return heapq.nsmallest(k, count, key=lambda w: (-count[w], w))


assert top_k_frequent_words(["i", "love", "leetcode", "i", "love", "coding"], 2) == [
    "i",
    "love",
]
words = ["the", "day", "is", "sunny", "the", "the", "the", "sunny", "is", "is"]
assert top_k_frequent_words(words, 4) == ["the", "is", "sunny", "day"]
print(top_k_frequent_words(words, 4))

['the', 'is', 'sunny', 'day']


### 5.1.6 [373. Find K Pairs with Smallest Sums](https://leetcode.com/problems/find-k-pairs-with-smallest-sums/) — ★★ · Medium

**Problem.** Sorted arrays `a`, `b`; the `k` pairs `(a[i], b[j])` with the smallest sums.  
**Signal.** Merge k sorted lists: row `i` is `a[i] + b[0], a[i] + b[1], …`.  
*New here: Seeing an implicit k-way merge (rows of a virtual matrix).*

**Fine print.** Ask before coding:
- Can `k` exceed `len(a) * len(b)`? → Yes, then return every pair.
- Duplicates? → Yes.

> 💡 **Key insight.** For a fixed `i`, sums `a[i] + b[j]` increase with `j`, so each `i` is a sorted list and the problem is "take the first k of a k-way merge".

**Brute force →** All m · n pairs, then sort → O(mn log mn). Heap merge → O(k log k).

**Steps**
1. Seed the heap with `(a[i] + b[0], i, 0)` for the first k values of `i`.
2. Pop the smallest; record the pair.
3. Push `(a[i] + b[j + 1], i, j + 1)` if it exists.
4. Stop after k pairs.

**Complexity**

- **Time:** O(k log k).
- **Space:** O(k).

In [65]:
def k_smallest_pairs(a: list[int], b: list[int], k: int) -> list[list[int]]:
    """k pairs with the smallest sums."""
    # head of each row i: pair (i, 0)
    heap = [(a[i] + b[0], i, 0) for i in range(min(k, len(a)))]
    out = []
    while heap and len(out) < k:
        _, i, j = heapq.heappop(heap)
        out.append([a[i], b[j]])
        # next pair in the same row
        if j + 1 < len(b):
            heapq.heappush(heap, (a[i] + b[j + 1], i, j + 1))
    return out


assert k_smallest_pairs([1, 7, 11], [2, 4, 6], 3) == [[1, 2], [1, 4], [1, 6]]
assert k_smallest_pairs([1, 1, 2], [1, 2, 3], 2) == [[1, 1], [1, 1]]
print(k_smallest_pairs([1, 7, 11], [2, 4, 6], 3))

[[1, 2], [1, 4], [1, 6]]


### 5.1.7 [264. Ugly Number II](https://leetcode.com/problems/ugly-number-ii/) — ★ · Medium

**Problem.** The n-th number whose only prime factors are 2, 3 and 5.  
**Signal.** Merge three sorted streams (`2×ugly`, `3×ugly`, `5×ugly`) with three pointers.  
*New here: A 3-way merge with pointers instead of a heap, and dedup by advancing all ties.*

**Fine print.** Ask before coding:
- Is 1 an ugly number? → Yes, the first one.
- Only primes 2, 3 and 5? → Yes.

> 💡 **Key insight.** Every ugly number is an earlier ugly number times 2, 3 or 5, so merging those three sorted streams generates them in order.

**Brute force →** Test every integer's factors → far too slow. Heap with a seen-set → O(n log n). Three pointers → O(n).

**Steps**
1. `ugly = [1]`, pointers `i2 = i3 = i5 = 0`.
2. Next = min of `ugly[i2]·2`, `ugly[i3]·3`, `ugly[i5]·5`.
3. Append it and advance *every* pointer that produced it.
4. Repeat until n numbers exist.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [66]:
def nth_ugly_number(n: int) -> int:
    """n-th 2-3-5 smooth number."""
    ugly = [1]
    # each pointer marks the next ugly number to multiply by its factor
    i2 = i3 = i5 = 0
    while len(ugly) < n:
        nxt = min(ugly[i2] * 2, ugly[i3] * 3, ugly[i5] * 5)
        ugly.append(nxt)
        # advance all producers so duplicates (6 = 2*3 = 3*2) appear once
        i2 += nxt == ugly[i2] * 2
        i3 += nxt == ugly[i3] * 3
        i5 += nxt == ugly[i5] * 5
    return ugly[-1]


assert nth_ugly_number(10) == 12
assert nth_ugly_number(1) == 1
print(nth_ugly_number(10))

12


## 5.2 Scheduling and simulation with heaps

Simulation questions: something becomes *available* over time, and you always act on the
best available item. Sort by availability, push items onto a heap as they become available,
pop the best.

### 5.2.1 [621. Task Scheduler](https://leetcode.com/problems/task-scheduler/) — ★★ · Medium

**Problem.** Tasks with a cooldown `n` between equal tasks; minimum time slots (idle allowed).  
**Signal.** The most frequent task dictates the frame → counting formula.  
*New here: Replacing a heap simulation with a counting argument.*

**Fine print.** Ask before coding:
- Can tasks run in any order? → Yes.
- Is the cooldown `n` between identical tasks only? → Yes.

> 💡 **Key insight.** The most frequent task needs `max_count − 1` full gaps of length `n + 1`, plus a last row for every task tied at the max; other tasks fill idle slots, and if there are more than enough, no idling is needed at all.

**Brute force →** Simulate time with a max-heap and a cooldown queue → O(N log 26). The formula → O(N).

**Steps**
1. Count tasks; `max_count` = highest count; `ties` = tasks with that count.
2. Frame length = `(max_count − 1) × (n + 1) + ties`.
3. Answer = `max(len(tasks), frame)`.

**Complexity**

- **Time:** O(N).
- **Space:** O(26).

In [67]:
def least_interval(tasks: list[str], n: int) -> int:
    """Fewest slots to run all tasks with cooldown n."""
    counts = Counter(tasks).values()
    max_count = max(counts)
    # tasks that also need a slot in the final row
    ties = sum(1 for c in counts if c == max_count)
    # frame forced by the busiest task, or no idle time at all
    return max(len(tasks), (max_count - 1) * (n + 1) + ties)


assert least_interval(list("AAABBB"), 2) == 8
assert least_interval(list("ACABDB"), 1) == 6
assert least_interval(list("AAABBB"), 3) == 10
print(least_interval(list("AAABBB"), 2))

8


### 5.2.2 [767. Reorganize String](https://leetcode.com/problems/reorganize-string/) — ★ · Medium

**Problem.** Rearrange so no two adjacent characters are equal, or return `""`.  
**Signal.** Always place the most frequent remaining character that isn't the previous one → max-heap.  
*New here: Holding an item out of the heap for one step (a cooldown of 1).*

**Fine print.** Ask before coding:
- Impossible? → Return `""`.
- Any valid arrangement? → Yes.

> 💡 **Key insight.** Placing the most frequent available character first is safest; holding the last-used one out of the heap for one round prevents placing it twice in a row.

**Brute force →** Try permutations → factorial. Greedy with a heap → O(n log 26).

**Steps**
1. Max-heap of `(−count, char)`.
2. Pop the top and append it.
3. Push back the previously held character (if it still has copies).
4. Hold the current one; if the output is shorter than `s` at the end, return `""`.

**Complexity**

- **Time:** O(n log 26).
- **Space:** O(26).

In [68]:
def reorganize_string(s: str) -> str:
    """Arrangement with no equal neighbours, or empty."""
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(heap)
    out: list[str] = []
    prev = None
    while heap:
        # most frequent character that is allowed now
        c, ch = heapq.heappop(heap)
        out.append(ch)
        # the character held last round may be used again
        if prev:
            heapq.heappush(heap, prev)
        # hold this one out for a round (if copies remain)
        prev = (c + 1, ch) if c + 1 < 0 else None
    # leftover held copies mean it was impossible
    return "".join(out) if len(out) == len(s) else ""


res = reorganize_string("aab")
assert res == "aba"
assert reorganize_string("aaab") == ""
assert all(
    a != b for a, b in zip(reorganize_string("aaabbc"), reorganize_string("aaabbc")[1:])
)
print(res)

aba


### 5.2.3 [355. Design Twitter](https://leetcode.com/problems/design-twitter/) — ★★ · Medium

**Problem.** `post_tweet`, `follow`, `unfollow`, and `get_news_feed` (10 most recent tweets from the user and followees).  
**Signal.** Merge k sorted tweet lists → heap over each followee's latest tweet.  
*New here: k-way merge inside a design question.*

**Fine print.** Ask before coding:
- Most recent 10, including the user's own tweets? → Yes.
- Can a user unfollow themselves? → Ignore it.
- Are tweet IDs unique? → Yes.

> 💡 **Key insight.** Each user's tweets are already in time order, so the feed is the first 10 items of a k-way merge over the followees' lists.

**Brute force →** Gather all followees' tweets and sort → O(T log T). Heap merge → O(F + 10 log F).

**Steps**
1. Store `(time, tweet_id)` per user; time decreases so smaller = newer.
2. Feed: seed a heap with each followee's (and own) newest tweet.
3. Pop up to 10 times, pushing that user's next older tweet.
4. Follow / unfollow update a set (no self-follow).

**Complexity**

- **Time:** O(F + 10 log F) per feed, O(1) post/follow.
- **Space:** O(total tweets + follows).

In [69]:
class Twitter:
    """Minimal Twitter feed."""

    def __init__(self) -> None:
        # decreasing counter: newer tweets compare smaller
        self.time = 0
        self.tweets: dict[int, list[tuple[int, int]]] = defaultdict(list)
        self.following: dict[int, set[int]] = defaultdict(set)

    def post_tweet(self, user_id: int, tweet_id: int) -> None:
        self.time -= 1
        self.tweets[user_id].append((self.time, tweet_id))

    def get_news_feed(self, user_id: int) -> list[int]:
        # newest tweet of each followed user (and the user)
        heap = []
        for u in self.following[user_id] | {user_id}:
            if self.tweets[u]:
                i = len(self.tweets[u]) - 1
                heap.append((self.tweets[u][i][0], u, i))
        heapq.heapify(heap)
        feed = []
        while heap and len(feed) < 10:
            _, u, i = heapq.heappop(heap)
            feed.append(self.tweets[u][i][1])
            # that user's next older tweet joins the merge
            if i:
                heapq.heappush(heap, (self.tweets[u][i - 1][0], u, i - 1))
        return feed

    def follow(self, follower: int, followee: int) -> None:
        if follower != followee:
            self.following[follower].add(followee)

    def unfollow(self, follower: int, followee: int) -> None:
        self.following[follower].discard(followee)


tw = Twitter()
tw.post_tweet(1, 5)
assert tw.get_news_feed(1) == [5]
tw.follow(1, 2)
tw.post_tweet(2, 6)
assert tw.get_news_feed(1) == [6, 5]
tw.unfollow(1, 2)
assert tw.get_news_feed(1) == [5]
print("ok")

ok


### 5.2.4 [295. Find Median from Data Stream](https://leetcode.com/problems/find-median-from-data-stream/) — ★★★ · Hard

**Problem.** `add_num` and `find_median` over a stream.  
**Signal.** Two heaps: max-heap for the lower half, min-heap for the upper half.  
*New here: Two heaps balanced against each other.*

**Fine print.** Ask before coding:
- Even count? → Return the mean of the two middle values.
- Follow-up: values in 0–100? → Use counting buckets.

> 💡 **Key insight.** The median sits between the largest of the lower half and the smallest of the upper half — exactly the tops of a max-heap and a min-heap.

**Brute force →** Insert into a sorted list → O(n) per add. Two heaps → O(log n) add, O(1) median.

**Steps**
1. Push the new number into the lower (max) heap.
2. Move the lower heap's largest into the upper heap (keeps every low ≤ every high).
3. If the upper heap is now bigger, move its smallest back.
4. Median = top of lower (odd count) or the average of both tops.

**Complexity**

- **Time:** O(log n) per add, O(1) per median.
- **Space:** O(n).

In [70]:
class MedianFinder:
    """Streaming median with two heaps."""

    def __init__(self) -> None:
        # low: max-heap (negated) of the smaller half; high: min-heap of the larger
        self.low: list[int] = []
        self.high: list[int] = []

    def add_num(self, num: int) -> None:
        heapq.heappush(self.low, -num)
        # move the largest low value up to keep the halves ordered
        heapq.heappush(self.high, -heapq.heappop(self.low))
        # rebalance: low keeps the extra element
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def find_median(self) -> float:
        if len(self.low) > len(self.high):
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2


mf = MedianFinder()
mf.add_num(1)
mf.add_num(2)
assert mf.find_median() == 1.5
mf.add_num(3)
assert mf.find_median() == 2.0
print(mf.find_median())

2.0


### 5.2.5 [502. IPO](https://leetcode.com/problems/ipo/) — ★★ · Hard

**Problem.** Start with capital `w`; do at most `k` projects, each needs `capital[i]` and yields `profits[i]`. Maximise final capital.  
**Signal.** Greedy: among affordable projects, always take the most profitable → sort by capital + max-heap of profits.  
*New here: The "unlock then pick best" pattern: sorted availability + heap of values.*

**Fine print.** Ask before coding:
- Is capital not spent when doing a project? → Correct, it only has to be reached.
- Can each project be done only once? → Yes.

> 💡 **Key insight.** Capital never decreases, so once a project is affordable it stays affordable; at every step the most profitable affordable project is the best choice.

**Brute force →** Scan all projects each round → O(k · n). Sort + heap → O((n + k) log n).

**Steps**
1. Sort projects by required capital.
2. Each round, push the profit of every newly affordable project onto a max-heap.
3. Pop the best profit and add it to `w`.
4. Stop early if nothing is affordable.

**Complexity**

- **Time:** O(n log n + k log n).
- **Space:** O(n).

In [71]:
def find_maximized_capital(
    k: int, w: int, profits: list[int], capital: list[int]
) -> int:
    """Capital after at most k greedy projects."""
    projects = sorted(zip(capital, profits))
    # max-heap of profits among affordable projects
    heap: list[int] = []
    i = 0
    for _ in range(k):
        # unlock everything the current capital can afford
        while i < len(projects) and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1])
            i += 1
        if not heap:
            break
        # take the most profitable one
        w -= heapq.heappop(heap)
    return w


assert find_maximized_capital(2, 0, [1, 2, 3], [0, 1, 1]) == 4
assert find_maximized_capital(3, 0, [1, 2, 3], [0, 1, 2]) == 6
print(find_maximized_capital(2, 0, [1, 2, 3], [0, 1, 1]))

4


### 5.2.6 [1834. Single-Threaded CPU](https://leetcode.com/problems/single-threaded-cpu/) — ★ · Medium

**Problem.** Tasks `[enqueue_time, processing_time]`; the CPU always picks the available task with the shortest processing time (ties by index). Return the processing order.  
**Signal.** Event simulation: sort by enqueue time + min-heap of available tasks.  
*New here: Simulating a clock, including jumping over idle time.*

**Fine print.** Ask before coding:
- Is the CPU idle if no task is available? → Yes, jump to the next enqueue time.
- Ties on processing time? → Smallest index.

> 💡 **Key insight.** At any moment only tasks already enqueued compete, and among them the CPU wants the shortest — a heap keyed by `(processing, index)`.

**Brute force →** Scan all tasks at each step for the best available → O(n²). Sort + heap → O(n log n).

**Steps**
1. Sort task indices by enqueue time.
2. If nothing is available, jump the clock to the next enqueue time.
3. Push every task enqueued by now onto the heap.
4. Pop and run the best; advance the clock by its duration.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

In [72]:
def get_order(tasks: list[list[int]]) -> list[int]:
    """Indices in the order the CPU processes them."""
    order = sorted(range(len(tasks)), key=lambda i: tasks[i][0])
    # available tasks keyed by (processing time, index)
    heap: list[tuple[int, int]] = []
    out, time, i = [], 0, 0
    while len(out) < len(tasks):
        # idle CPU: jump to the next arrival
        if not heap and time < tasks[order[i]][0]:
            time = tasks[order[i]][0]
        # everything that has arrived joins the queue
        while i < len(order) and tasks[order[i]][0] <= time:
            heapq.heappush(heap, (tasks[order[i]][1], order[i]))
            i += 1
        # run the shortest available task to completion
        duration, idx = heapq.heappop(heap)
        time += duration
        out.append(idx)
    return out


assert get_order([[1, 2], [2, 4], [3, 2], [4, 1]]) == [0, 2, 3, 1]
assert get_order([[7, 10], [7, 12], [7, 5], [7, 4], [7, 2]]) == [4, 3, 2, 0, 1]
print(get_order([[1, 2], [2, 4], [3, 2], [4, 1]]))

[0, 2, 3, 1]


### 5.2.7 [1642. Furthest Building You Can Reach](https://leetcode.com/problems/furthest-building-you-can-reach/) — ★ · Medium

**Problem.** Climbing up needs bricks (height difference) or one ladder. How far can you go?  
**Signal.** Use ladders on the largest climbs → min-heap of the climbs currently given ladders.  
*New here: "Regret" greedy: decide tentatively, and undo the worst decision later.*

**Fine print.** Ask before coding:
- Is going down free? → Yes.
- Can a ladder cover any height? → Yes.
- 0-indexed answer? → Yes, the index of the furthest building.

> 💡 **Key insight.** Ladders are worth most on the biggest climbs; give every climb a ladder for now, and when ladders run out, pay bricks for the *smallest* laddered climb.

**Brute force →** Try every assignment of ladders → exponential; binary search on the answer + sort → O(n log² n). Regret heap → O(n log L).

**Steps**
1. For each upward climb, push it onto a min-heap (tentative ladder).
2. If the heap holds more climbs than ladders, pop the smallest and pay it in bricks.
3. If bricks go negative, the current building is the last reachable.
4. Otherwise you reach the end.

**Complexity**

- **Time:** O(n log L) — heap size ≤ ladders + 1.
- **Space:** O(L).

In [73]:
def furthest_building(heights: list[int], bricks: int, ladders: int) -> int:
    """Furthest reachable index."""
    # climbs currently covered by ladders (smallest on top)
    heap: list[int] = []
    for i in range(len(heights) - 1):
        climb = heights[i + 1] - heights[i]
        if climb <= 0:
            continue
        heapq.heappush(heap, climb)
        # out of ladders: the smallest laddered climb is paid in bricks
        if len(heap) > ladders:
            bricks -= heapq.heappop(heap)
            if bricks < 0:
                return i
    return len(heights) - 1


assert furthest_building([4, 2, 7, 6, 9, 14, 12], 5, 1) == 4
assert furthest_building([4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2) == 7
assert furthest_building([14, 3, 19, 3], 17, 0) == 3
print(furthest_building([4, 2, 7, 6, 9, 14, 12], 5, 1))

4


### 5.2.8 [632. Smallest Range Covering Elements from K Lists](https://leetcode.com/problems/smallest-range-covering-elements-from-k-lists/) — ★★ · Hard

**Problem.** Smallest range `[a, b]` containing at least one number from each of `k` sorted lists.  
**Signal.** Sliding a window across k sorted lists → min-heap of current heads + running max.  
*New here: A sliding window over k lists, with a heap for the minimum and a variable for the maximum.*

**Fine print.** Ask before coding:
- Ties? → Choose the range with the smaller start.
- Are the lists non-empty and sorted? → Yes.

> 💡 **Key insight.** One pointer per list always covers every list; the range is `[min head, max head]`, and only advancing the minimum's list can make it smaller.

**Brute force →** Merge everything with list labels and run a sliding window → O(N log N). Heap of heads → O(N log k).

**Steps**
1. Heap of each list's first element; `hi` = max of them.
2. Pop the minimum `lo`; record `[lo, hi]` if it is tighter.
3. Advance that list's pointer; push the next value and update `hi`.
4. Stop when a list runs out.

**Complexity**

- **Time:** O(N log k).
- **Space:** O(k).

In [74]:
def smallest_range(nums: list[list[int]]) -> list[int]:
    """Tightest range touching every list."""
    # current head of each list
    heap = [(row[0], i, 0) for i, row in enumerate(nums)]
    heapq.heapify(heap)
    hi = max(row[0] for row in nums)
    best = [heap[0][0], hi]
    while True:
        # the minimum head bounds the range from below
        lo, i, j = heapq.heappop(heap)
        if hi - lo < best[1] - best[0]:
            best = [lo, hi]
        # this list is exhausted: no range can cover all lists anymore
        if j + 1 == len(nums[i]):
            return best
        # advance the list that held the minimum
        nxt = nums[i][j + 1]
        hi = max(hi, nxt)
        heapq.heappush(heap, (nxt, i, j + 1))


lists = [[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]
assert smallest_range(lists) == [20, 24]
assert smallest_range([[1, 2, 3], [1, 2, 3], [1, 2, 3]]) == [1, 1]
print(smallest_range(lists))

[20, 24]


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Bottom-up DFS | height, diameter, balanced, max path, LCA | combine child results, update global best | O(n) |
| Tree DP (state pair) | rob/skip, cameras, coins | return a tuple of states per node | O(n) |
| Top-down DFS | path sums, good nodes, bounds | pass state down, backtrack shared lists | O(n) |
| Prefix sum on paths | downward paths summing to k | hash map of prefix sums, undo on return | O(n) |
| Construction | pre/in, in/post, serialise | root from one order, split by the other | O(n) |
| BFS by level | level lists, right view, width, completeness | `for _ in range(len(queue))` | O(n) |
| BST bounds / in-order | validate, k-th, successor, recover | `(low, high)` window; in-order is sorted | O(h)–O(n) |
| Trie | prefix queries, many words on one grid, max XOR | nested dicts + end marker | O(L) |
| Heap size k | top-k, k-th, k closest | min-heap for largest, max-heap for smallest | O(n log k) |
| Two heaps | streaming median | lower max-heap + upper min-heap | O(log n) |
| k-way merge | merge lists, smallest pairs, smallest range, feed | heap of current heads | O(N log k) |
| Greedy + heap | IPO, ladders, CPU scheduling, reorganise | sort by availability, heap by value | O(n log n) |